# EXL × Trinity Hackathon · Challenge 01 — Agentic healthcare integration

**What this notebook is:** a working prototype of an AI agent that connects three mock hospital systems. The agent **inspects** each system, **designs** how its fields map into FHIR (the healthcare data standard), **writes the integration code**, tests it and **fixes its own mistakes**. A person approves every stage before the next one runs.

**The end result:** one common API that returns everything about a patient (registration details, appointments and lab results) as a single FHIR record, plus measured numbers to compare with manual work.

### How to run it
1. **Test run (no API key needed):** leave `MODE` as `replay` and choose **Runtime → Run all**. It deliberately **stops at each ✅ review cell**, because the agent can't continue without human sign-off. Type your name, tick **APPROVED**, then choose **Runtime → Run after** to continue. It should reach the end with all checks passing.
2. **Real run:** get a free Gemini API key from **Google AI Studio**, set `MODE` to `live` in cell 1, and run the cells **one by one**. At each ✅ review cell, read the agent's output first, then approve. The time you spend reviewing is measured automatically, so this is the run to use for the timing comparison and the demo video.
3. To use Person 1's data, upload their files to the `team_data` folder (📁 icon on the left) and set `DATA_SOURCE` to `team`.
4. At the end, download the `outputs` zip. It has every file for the deck, the repo and Person 3's comparison.

### What Person 1 should send
| File | Format |
|---|---|
| `patients.json` | A list of patient records, as the patient API would return them (or set `TEAM_PATIENT_API_URL` if the API is running online and serves `GET /patients`) |
| `hospital.db` | A SQLite database file (a `.sql` script of `CREATE TABLE` + `INSERT` statements also works) |
| `lab_results.csv` | One row per test result, with a header row |
| `field_descriptions.md` | Optional. Short notes on what the fields mean. The agent can work without it |

*All data is synthetic. Never put real patient data into this notebook.*

In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown **How the agent thinks:** `live` calls Google Gemini (free API key needed). `replay` needs no key and replays saved agent outputs, so use it as the demo backup.
MODE = "replay"  #@param ["live", "replay"]
#@markdown Paste the Gemini API key here, or (safer) add it in Colab's 🔑 **Secrets** panel with the name `GEMINI_API_KEY`.
GEMINI_API_KEY = ""  #@param {type:"string"}
#@markdown Leave as `auto` to pick the newest Gemini Flash model automatically.
GEMINI_MODEL = "auto"  #@param {type:"string"}
#@markdown **Which data:** `placeholder` = built-in synthetic sample · `team` = Person 1's files in the `team_data` folder.
DATA_SOURCE = "team"  #@param ["placeholder", "team"]
TEAM_PATIENTS_JSON = "team_data/patients.json"  #@param {type:"string"}
TEAM_PATIENT_API_URL = ""  #@param {type:"string"}
TEAM_HOSPITAL_DB = "team_data/hospital.db"  #@param {type:"string"}
TEAM_LAB_CSV = "team_data/lab_results.csv"  #@param {type:"string"}
TEAM_FIELD_DESCRIPTIONS = "team_data/field_descriptions.md"  #@param {type:"string"}
#@markdown How many times the agent may try to fix its own code when a test fails:
MAX_FIX_ATTEMPTS = 3  #@param {type:"integer"}

if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass
if MODE == "live" and not GEMINI_API_KEY:
    raise RuntimeError("MODE is 'live' but no Gemini API key was found. Paste the key above (or add it to Colab Secrets), or switch MODE to 'replay'.")
if MODE == "replay" and DATA_SOURCE == "team":
    print("ℹ️ Replay with team data uses the built-in example outputs prepared for the team's Synthea sample (team_data from the build script).")
print(f"Mode: {MODE} · Data: {DATA_SOURCE}")

In [ ]:
#@title 2 · Toolkit: logging, privacy masking and review gates (run it, no need to read) { display-mode: "form" }
import json, os, re, csv, time, sqlite3, ast, shutil, datetime as dt
import requests
import pandas as pd
try:
    from IPython.display import display, Markdown
except ImportError:  # outside Jupyter/Colab
    display = print
    def Markdown(text):
        return text

pd.set_option("display.max_colwidth", 90)
OUT = "outputs"
for folder in (OUT, f"{OUT}/agent_cache", f"{OUT}/generated", "data", "team_data"):
    os.makedirs(folder, exist_ok=True)
AUDIT_FILE = f"{OUT}/audit_log.jsonl"
RUN_ID = dt.datetime.now().strftime("%Y%m%d-%H%M%S")

# ---------- privacy ----------
NAME_COLUMNS = {"name", "full_name", "fullname", "first_name", "firstname", "last_name", "lastname", "surname",
                "given_name", "family_name", "forename", "patient_name", "pt_name", "pat_name"}
PHI_COLUMN_PATTERN = re.compile(r"dob|birth|phone|mobile|tel$|address|addr|email|ppsn|ssn|passport|drivers|eircode|postcode|zip", re.I)
PHI_VALUES = set()   # filled in when the source systems load; used to scan logs for leaks

def is_phi_column(column):
    column = str(column).strip()
    return column.lower() in NAME_COLUMNS or bool(PHI_COLUMN_PATTERN.search(column))

def shape_mask(value):
    """Hide the content but keep the format: 'Aoife Murphy' -> 'Xxxxx Xxxxxx', '12/04/1985' -> '99/99/9999'."""
    if value is None:
        return None
    return "".join("9" if c.isdigit() else ("X" if c.isupper() else "x") if c.isalpha() else c for c in str(value))

def mask_record(record):
    """Mask every field whose column name looks like personal information."""
    return {k: (shape_mask(v) if is_phi_column(k) else v) for k, v in record.items()}

def remember_phi(records):
    for record in records:
        for key, value in record.items():
            if is_phi_column(key) and value not in (None, ""):
                text = str(value).strip()
                if len(text) >= 4:
                    PHI_VALUES.add(text)
                if key.lower() in NAME_COLUMNS:
                    PHI_VALUES.update(part for part in re.split(r"\s+", text) if len(part) >= 4)

_phi_pattern = {"size": -1, "regex": None}

def _phi_regex():
    """One pattern matching any known personal value as a whole word (not inside IDs)."""
    if _phi_pattern["size"] != len(PHI_VALUES):
        alternatives = "|".join(re.escape(v) for v in sorted(PHI_VALUES, key=len, reverse=True))
        _phi_pattern.update(size=len(PHI_VALUES), regex=re.compile(rf"(?<![A-Za-z0-9])(?:{alternatives})(?![A-Za-z0-9])") if alternatives else None)
    return _phi_pattern["regex"]

def contains_phi(text):
    regex = _phi_regex()
    return bool(regex and regex.search(str(text)))

def redact(text):
    regex = _phi_regex()
    return regex.sub(lambda m: shape_mask(m.group(0)), str(text)) if regex else str(text)

# ---------- audit log ----------
def audit(event, **details):
    entry = {"time": dt.datetime.now().isoformat(timespec="seconds"), "run": RUN_ID, "event": event, **details}
    with open(AUDIT_FILE, "a") as handle:
        handle.write(json.dumps(entry, default=str) + "\n")
    return entry

# ---------- human review gates ----------
STAGES = {}

def stage_finished(stage, **summary):
    STAGES[stage] = {"finished_at": time.time(), "approved": False, **summary}
    audit("stage_finished", stage=stage, **summary)

def approve(stage, reviewer, approved, notes=""):
    if stage not in STAGES:
        raise RuntimeError(f"Run the '{stage}' stage before reviewing it.")
    if not approved or not str(reviewer).strip():
        raise RuntimeError(f"Not approved yet. Check the '{stage}' output above, type your name in REVIEWER, tick APPROVED, then run this cell again.")
    review_seconds = round(time.time() - STAGES[stage]["finished_at"], 1)
    STAGES[stage].update(approved=True, reviewer=str(reviewer).strip(), review_seconds=review_seconds, notes=notes)
    audit("human_approval", stage=stage, reviewer=str(reviewer).strip(), review_seconds=review_seconds, notes=notes)
    print(f"✅ '{stage}' approved by {str(reviewer).strip()} after {review_seconds / 60:.1f} min of review. The next stage is unlocked.")

def require_approved(stage):
    if not STAGES.get(stage, {}).get("approved"):
        raise RuntimeError(f"🔒 Blocked: the '{stage}' stage has not been approved by a human reviewer yet. The agent cannot continue without sign-off.")

# ---------- small helpers ----------
def get_path(obj, path):
    """Read 'name[0].family' style paths from a nested record."""
    current = obj
    for part in re.findall(r"[^.\[\]]+", path):
        if isinstance(current, list):
            if not part.isdigit() or int(part) >= len(current):
                return None
            current = current[int(part)]
        elif isinstance(current, dict):
            current = current.get(part)
        else:
            return None
        if current is None:
            return None
    return current

def parse_json(text):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise

def heading(text):
    display(Markdown(f"#### {text}"))

print("Toolkit ready.")

In [ ]:
#@title 3 · The three mock hospital systems { display-mode: "form" }
#@markdown The placeholder data is synthetic (invented) and deliberately messy, like real legacy systems. To use Person 1's files instead, set `DATA_SOURCE` to `team` in cell 1.

PLACEHOLDER_PATIENTS = [
    {"patient_id": "P001", "full_name": "Aoife Murphy", "dob": "12/04/1985", "sex": "F", "mobile": "087 555 0101", "county": "Dublin", "registered_on": "2019-03-02"},
    {"patient_id": "P002", "full_name": "Ciarán O'Brien", "dob": "03/11/1972", "sex": "M", "mobile": "086 555 0102", "county": "Cork", "registered_on": "2015-07-19"},
    {"patient_id": "P003", "full_name": "Niamh Kelly", "dob": "25/07/1990", "sex": "F", "mobile": "085 555 0103", "county": "Galway", "registered_on": "2021-01-11"},
    {"patient_id": "P004", "full_name": "Seán Byrne", "dob": "14/02/1958", "sex": "M", "mobile": "087 555 0104", "county": "Dublin", "registered_on": "2012-10-30"},
    {"patient_id": "P005", "full_name": "Priya Nair", "dob": "30/09/1988", "sex": "F", "mobile": "083 555 0105", "county": "Kildare", "registered_on": "2022-05-06"},
    {"patient_id": "P006", "full_name": "Tomasz Nowak", "dob": "08/06/1979", "sex": "M", "mobile": "089 555 0106", "county": "Dublin", "registered_on": "2018-02-14"},
    {"patient_id": "P007", "full_name": "Grace Walsh", "dob": "19/12/2001", "sex": "F", "mobile": "087 555 0107", "county": "Wicklow", "registered_on": "2023-09-01"},
    {"patient_id": "P008", "full_name": "Oluwaseun Adeyemi", "dob": "22/03/1995", "sex": "M", "mobile": "086 555 0108", "county": "Meath", "registered_on": "2020-11-23"},
]
PLACEHOLDER_APPOINTMENTS = [
    (1001, "P001", "20261120 1030", "CARD", "B", "C17"), (1002, "P001", "20260915 0900", "GEN", "A", "C02"),
    (1003, "P002", "20261118 1400", "ENDO", "B", "C09"), (1004, "P002", "20260801 1115", "ENDO", "A", "C09"),
    (1005, "P003", "20261125 0930", "ORTH", "B", "C21"), (1006, "P004", "20261021 1600", "CARD", "N", "C17"),
    (1007, "P004", "20261202 1000", "RESP", "B", "C33"), (1008, "P005", "20261103 1130", "GEN", "C", "C02"),
    (1009, "P005", "20261117 1130", "GEN", "B", "C02"), (1010, "P006", "20260930 0845", "ORTH", "A", "C21"),
    (1011, "P007", "20261130 1500", "RESP", "B", "C33"), (1012, "P008", "20261110 1015", "ENDO", "A", "C09"),
    (1013, "P009", "20261119 1200", "CARD", "B", "C17"),
]
PLACEHOLDER_DEPARTMENTS = [("CARD", "Cardiology"), ("ENDO", "Endocrinology"), ("GEN", "General Medicine"),
                           ("ORTH", "Orthopaedics"), ("RESP", "Respiratory")]
PLACEHOLDER_LABS = [
    ["patient_ref", "sample_date", "test_code", "test_name", "result", "units", "ref_range"],
    ["P001", "02-11-2026", "718-7", "Haemoglobin", "13.5", "g/dL", "12.0-15.5"],
    ["P001", "02-11-2026", "2093-3", "Total cholesterol", "5.8", "mmol/L", "0.0-5.0"],
    ["P002", "28-10-2026", "4548-4", "HbA1c", "7.9", "%", "4.0-5.6"],
    ["P002", "28-10-2026", "2345-7", "Glucose (fasting)", "8.4", "mmol/L", "3.9-5.6"],
    ["P003", "05-11-2026", "718-7", "Haemoglobin", "11.2", "g/dL", "12.0-15.5"],
    ["P004", "14-10-2026", "2160-0", "Creatinine", "96", "umol/L", "45-90"],
    ["P004", "14-10-2026", "2093-3", "Total cholesterol", "6.3", "mmol/L", "0.0-5.0"],
    ["P005", "30-10-2026", "718-7", "Haemoglobin", "12.9", "g/dL", "12.0-15.5"],
    ["P006", "22-09-2026", "2345-7", "Glucose (fasting)", "5.1", "mmol/L", "3.9-5.6"],
    ["P007", "09-11-2026", "718-7", "Haemoglobin", "14.1", "g/dL", "12.0-15.5"],
    ["P008", "03-11-2026", "4548-4", "HbA1c", "5.4", "%", "4.0-5.6"],
    ["", "04-11-2026", "2345-7", "Glucose (fasting)", "6.0", "mmol/L", "3.9-5.6"],
    ["P003", "05-11-2026", "2160-0", "Creatinine", "pending", "umol/L", "45-90"],
]
PLACEHOLDER_DOCS = {
    "patient_api": "Patient Registration API. GET /patients returns all registered patients; GET /patients/{patient_id} returns one. JSON. dob = date of birth (day first). sex = F/M/U.",
    "hospital_db": "No documentation exists for this legacy database. Only the schema is available.",
    "lab_csv": "LabSys nightly export. One row per test result. test_code uses LOINC. sample_date is the collection date.",
}

if DATA_SOURCE == "placeholder":
    PATIENTS_FILE, DB_FILE, LAB_FILE = "data/patients.json", "data/hospital.db", "data/lab_results.csv"
    with open(PATIENTS_FILE, "w") as handle:
        json.dump(PLACEHOLDER_PATIENTS, handle, indent=2, ensure_ascii=False)
    if os.path.exists(DB_FILE):
        os.remove(DB_FILE)
    con = sqlite3.connect(DB_FILE)
    con.execute("CREATE TABLE TBL_APPT (APPT_NO INTEGER PRIMARY KEY, PT_ID TEXT, APPT_DT TEXT, DEPT_CD TEXT, STAT TEXT, CLIN_ID TEXT)")
    con.execute("CREATE TABLE LKP_DEPT (DEPT_CD TEXT PRIMARY KEY, DEPT_DESC TEXT)")
    con.executemany("INSERT INTO TBL_APPT VALUES (?,?,?,?,?,?)", PLACEHOLDER_APPOINTMENTS)
    con.executemany("INSERT INTO LKP_DEPT VALUES (?,?)", PLACEHOLDER_DEPARTMENTS)
    con.commit(); con.close()
    with open(LAB_FILE, "w", newline="", encoding="utf-8") as handle:
        csv.writer(handle).writerows(PLACEHOLDER_LABS)
    DOCS = dict(PLACEHOLDER_DOCS)
else:
    PATIENTS_FILE, LAB_FILE = TEAM_PATIENTS_JSON, TEAM_LAB_CSV
    if TEAM_HOSPITAL_DB.lower().endswith(".sql"):
        DB_FILE = "data/team_hospital.db"
        if os.path.exists(DB_FILE):
            os.remove(DB_FILE)
        con = sqlite3.connect(DB_FILE)
        con.executescript(open(TEAM_HOSPITAL_DB, encoding="utf-8").read())
        con.commit(); con.close()
    else:
        DB_FILE = TEAM_HOSPITAL_DB
    for label, path in (("patients JSON", PATIENTS_FILE), ("hospital database", DB_FILE), ("lab CSV", LAB_FILE)):
        if label == "patients JSON" and TEAM_PATIENT_API_URL:
            continue
        if not os.path.exists(path):
            raise FileNotFoundError(f"Couldn't find the {label} at '{path}'. Upload it to the team_data folder or fix the path in cell 1.")
    DOCS = {"patient_api": "", "hospital_db": "", "lab_csv": ""}
    if os.path.exists(TEAM_FIELD_DESCRIPTIONS):
        DOCS["shared"] = open(TEAM_FIELD_DESCRIPTIONS, encoding="utf-8").read()

_patient_store = []
if not (DATA_SOURCE == "team" and TEAM_PATIENT_API_URL):
    with open(PATIENTS_FILE, encoding="utf-8") as handle:
        loaded = json.load(handle)
    if isinstance(loaded, dict):  # e.g. {"patients": [...]}
        loaded = next((v for v in loaded.values() if isinstance(v, list)), [])
    _patient_store = loaded

def api_get(path):
    """The Patient Registration System's REST API (GET only)."""
    if DATA_SOURCE == "team" and TEAM_PATIENT_API_URL:
        response = requests.get(TEAM_PATIENT_API_URL.rstrip("/") + path, timeout=30)
        response.raise_for_status()
        return response.json()
    if path.rstrip("/") == "/patients":
        return [dict(p) for p in _patient_store]
    match = re.fullmatch(r"/patients/([^/]+)", path)
    if match:
        for patient in _patient_store:
            if str(patient.get("patient_id", patient.get("id"))) == match.group(1):
                return dict(patient)
        raise LookupError(f"404 Not Found: {path}")
    raise LookupError(f"404 Not Found: {path}")

SYSTEM_IDS = ["patient_api", "hospital_db", "lab_csv"]
SYSTEM_LABELS = {"patient_api": "Patient registration system (REST API)",
                 "hospital_db": "Hospital database (SQL)",
                 "lab_csv": "Laboratory system (CSV file)"}

def open_source(system_id):
    """What each generated adapter receives as `source`."""
    if system_id == "patient_api":
        return api_get, None
    if system_id == "hospital_db":
        connection = sqlite3.connect(DB_FILE)
        return connection, connection
    return LAB_FILE, None

# Remember personal values so we can prove they never reach the logs
remember_phi(api_get("/patients"))
_con = sqlite3.connect(DB_FILE)
for (_table,) in _con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'").fetchall():
    _cur = _con.execute(f'SELECT * FROM "{_table}"')
    _cols = [c[0] for c in _cur.description]
    remember_phi([dict(zip(_cols, row)) for row in _cur.fetchall()])
_con.close()
with open(LAB_FILE, newline="", encoding="utf-8-sig") as handle:
    remember_phi(list(csv.DictReader(handle)))

heading("Patient registration system: REST API · GET /patients")
display(pd.DataFrame(api_get("/patients")).head(10))
heading("Hospital database: SQL")
_con = sqlite3.connect(DB_FILE)
for (_table,) in _con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'").fetchall():
    print(f"Table {_table}")
    display(pd.read_sql_query(f'SELECT * FROM "{_table}" LIMIT 8', _con))
_con.close()
heading("Laboratory system: CSV export")
display(pd.read_csv(LAB_FILE, dtype=str, keep_default_na=False).head(15))
audit("sources_loaded", data_source=DATA_SOURCE)

In [ ]:
#@title 4 · The agreed output: FHIR-aligned structure { display-mode: "form" }
#@markdown This is the "common language" every system gets translated into. The team agrees this up front, and the agent must stick to it.
TARGET = {
    "Patient": {
        "fields": {"id": "the patient's ID from the registration system",
                   "name[0].family": "surname", "name[0].given": "list of first names",
                   "gender": "male | female | other | unknown", "birthDate": "YYYY-MM-DD"},
        "required": ["id", "name[0].family", "gender", "birthDate"],
        "example": {"resourceType": "Patient", "id": "P123", "name": [{"family": "Doe", "given": ["Jane"]}],
                    "gender": "female", "birthDate": "1970-01-31"}},
    "Appointment": {
        "fields": {"id": "'APPT-' + source appointment number",
                   "status": "proposed | pending | booked | arrived | fulfilled | cancelled | noshow | checked-in | waitlist",
                   "start": "YYYY-MM-DDTHH:MM:SS", "serviceType[0].text": "department name in words",
                   "participant[0].actor.reference": "'Patient/' + patient ID"},
        "required": ["id", "status", "start", "participant[0].actor.reference"],
        "example": {"resourceType": "Appointment", "id": "APPT-1", "status": "booked", "start": "2026-01-31T09:00:00",
                    "serviceType": [{"text": "Dermatology"}],
                    "participant": [{"actor": {"reference": "Patient/P123"}, "status": "accepted"}]}},
    "Encounter": {
        "fields": {"id": "'ENC-' + source admission number", "status": "planned | in-progress | finished | cancelled",
                   "period.start": "YYYY-MM-DDTHH:MM:SS", "period.end": "YYYY-MM-DDTHH:MM:SS (optional)",
                   "serviceType.text": "ward or department", "subject.reference": "'Patient/' + patient ID"},
        "required": ["id", "status", "period.start", "subject.reference"],
        "example": {"resourceType": "Encounter", "id": "ENC-1", "status": "finished", "period": {"start": "2026-01-02T08:00:00", "end": "2026-01-05T12:00:00"},
                    "serviceType": {"text": "Ward 3"}, "subject": {"reference": "Patient/P123"}}},
    "Observation": {
        "fields": {"id": "'LAB-' + source row or result number", "status": "final | preliminary | amended",
                   "code.coding[0].system": "http://loinc.org", "code.coding[0].code": "LOINC code",
                   "code.text": "test name in words", "subject.reference": "'Patient/' + patient ID",
                   "effectiveDateTime": "YYYY-MM-DD or YYYY-MM-DDTHH:MM:SS", "valueQuantity.value": "number",
                   "valueQuantity.unit": "unit text", "referenceRange[0].text": "normal range as text"},
        "required": ["id", "status", "code.coding[0].code", "subject.reference", "effectiveDateTime", "valueQuantity.value", "valueQuantity.unit"],
        "example": {"resourceType": "Observation", "id": "LAB-0001", "status": "final",
                    "code": {"coding": [{"system": "http://loinc.org", "code": "718-7", "display": "Haemoglobin"}], "text": "Haemoglobin"},
                    "subject": {"reference": "Patient/P123"}, "effectiveDateTime": "2026-01-31",
                    "valueQuantity": {"value": 13.2, "unit": "g/dL"}, "referenceRange": [{"text": "12.0-15.5"}]}},
}
SHARED_RULES = [
    "Every record carries 'resourceType' and a stable 'id' built from the source system's own key.",
    "Links to a patient are written as {'reference': 'Patient/<patient id>'} using the registration system's ID.",
    "Dates are ISO 8601: YYYY-MM-DD, date-times YYYY-MM-DDTHH:MM:SS.",
    "Coded values (status, gender) use only the allowed words listed in the target.",
    "Personal fields that have no target field (phone, address) are not copied.",
    "A record that fails validation is rejected and reported; it is never half-filled or guessed.",
]
DATE_FIELDS = {"Patient": ["birthDate"], "Appointment": ["start"], "Encounter": ["period.start", "period.end"], "Observation": ["effectiveDateTime"]}
REFERENCE_FIELDS = {"Appointment": ["participant[0].actor.reference"], "Encounter": ["subject.reference"], "Observation": ["subject.reference"]}
NUMBER_FIELDS = {"Observation": ["valueQuantity.value"]}
ALLOWED_VALUES = {
    ("Patient", "gender"): {"male", "female", "other", "unknown"},
    ("Appointment", "status"): {"proposed", "pending", "booked", "arrived", "fulfilled", "cancelled", "noshow", "entered-in-error", "checked-in", "waitlist"},
    ("Encounter", "status"): {"planned", "in-progress", "on-hold", "discharged", "completed", "finished", "cancelled", "entered-in-error", "unknown"},
    ("Observation", "status"): {"registered", "preliminary", "final", "amended", "corrected", "cancelled", "entered-in-error", "unknown"},
}
TARGET_FOR_PROMPT = {name: {"fields": spec["fields"], "required": spec["required"], "example": spec["example"]} for name, spec in TARGET.items()}
rows = [{"resource": name, "field": field, "meaning / format": meaning, "required": field in spec["required"]}
        for name, spec in TARGET.items() for field, meaning in spec["fields"].items()]
display(pd.DataFrame(rows))
print("Shared rules every generated integration must follow:")
for rule in SHARED_RULES:
    print(" •", rule)

In [ ]:
#@title 5 · Connect the agent to its AI model { display-mode: "form" }
API_ROOT = "https://generativelanguage.googleapis.com/v1beta"
MODEL_NAME = None
REPLAY_EXAMPLES = globals().get("REPLAY_EXAMPLES", {})  # built-in backup outputs, loaded by cell 5b

def pick_model():
    """Choose the newest stable Gemini 'Flash' model this key can use."""
    if GEMINI_MODEL and GEMINI_MODEL != "auto":
        return GEMINI_MODEL
    response = requests.get(f"{API_ROOT}/models", headers={"x-goog-api-key": GEMINI_API_KEY}, params={"pageSize": 200}, timeout=30)
    response.raise_for_status()
    skip = ("lite", "tts", "image", "audio", "live", "embed", "native", "exp", "thinking", "8b", "learnlm", "robotics", "computer")
    best = None
    for model in response.json().get("models", []):
        name = model["name"].split("/")[-1]
        if "generateContent" not in model.get("supportedGenerationMethods", []):
            continue
        if not name.startswith("gemini-") or "flash" not in name or any(word in name for word in skip):
            continue
        version = re.search(r"gemini-(\d+(?:\.\d+)?)", name)
        if not version:
            continue
        rank = ("preview" not in name, float(version.group(1)), -len(name))
        if best is None or rank > best[0]:
            best = (rank, name)
    return best[1] if best else "gemini-2.5-flash"

def _call_gemini(prompt):
    url = f"{API_ROOT}/models/{MODEL_NAME}:generateContent"
    body = {"contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": 0.1, "responseMimeType": "application/json"}}
    for attempt in range(5):
        response = requests.post(url, headers={"x-goog-api-key": GEMINI_API_KEY}, json=body, timeout=300)
        if response.status_code in (429, 500, 502, 503, 504):
            wait = 15 * (attempt + 1)
            print(f"   (the model is busy, retrying in {wait}s)")
            time.sleep(wait)
            continue
        if response.status_code >= 400:
            raise RuntimeError(f"Gemini error {response.status_code}: {response.text[:400]}")
        parts = response.json()["candidates"][0]["content"]["parts"]
        return "".join(part.get("text", "") for part in parts if not part.get("thought"))
    raise RuntimeError("Gemini kept saying it's busy. Wait a minute and re-run this cell, or switch MODE to 'replay'.")

def ask_agent(task, prompt):
    """One step of agent work: send instructions + evidence to the AI model, get structured JSON back."""
    cache_file = f"{OUT}/agent_cache/{task}.json"
    started = time.time()
    if MODE == "live":
        text = _call_gemini(prompt)
        try:
            result = parse_json(text)
        except Exception:
            result = parse_json(_call_gemini(prompt + "\n\nYour previous reply was not valid JSON. Reply with valid JSON only."))
        with open(cache_file, "w") as handle:
            json.dump(result, handle, indent=2)
        source = f"live · {MODEL_NAME}"
    elif os.path.exists(cache_file):
        result = json.load(open(cache_file))
        source = "replay · saved output from an earlier live run"
    elif task in REPLAY_EXAMPLES:
        result = json.loads(json.dumps(REPLAY_EXAMPLES[task]))
        source = "replay · built-in example output"
    else:
        raise RuntimeError(f"No saved output for '{task}'. Run this stage once in live mode first.")
    if isinstance(result, list):  # the model sometimes wraps its JSON object in a list
        dicts = [d for d in result if isinstance(d, dict)]
        if len(dicts) == 1:
            result = dicts[0]
        elif dicts and all("target" in d for d in dicts):
            result = {"mappings": dicts}
        else:
            result = {k: v for d in dicts for k, v in d.items()}
    seconds = round(time.time() - started, 1)
    audit("agent_call", task=task, mode=MODE, source=source, seconds=seconds, prompt_characters=len(prompt))
    print(f"🤖 {task}: {source} · {seconds}s")
    return result, seconds

if MODE == "live":
    MODEL_NAME = pick_model()
    print(f"Agent model: {MODEL_NAME}")
else:
    print("Replay mode: the agent will replay saved outputs (no API calls).")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 5b · Backup outputs for the demo (built-in examples; no need to read) { display-mode: "form" }
#@markdown Example agent outputs for the **placeholder** data and for the **team** Synthea sample, so the notebook can run end to end without an API key. A live run saves its own outputs in `outputs/agent_cache`, and replay mode prefers those.
REPLAY_SETS = json.loads(r'''{
 "placeholder": {
  "discover": {
   "systems": [
    {
     "system_id": "patient_api",
     "summary": "Patient registration system. One record per registered patient with identity and contact details.",
     "access_method": "REST API, GET /patients (list) and GET /patients/{patient_id} (one), JSON responses",
     "entities": [
      {
       "entity": "/patients",
       "fields": [
        {
         "field": "patient_id",
         "type": "string",
         "meaning": "Unique patient identifier used across the hospital",
         "format": "P + 3 digits, e.g. P999",
         "personal_data": false,
         "patient_identifier": true
        },
        {
         "field": "full_name",
         "type": "string",
         "meaning": "Patient's full name, first name(s) then surname",
         "format": "Xxxxx Xxxxxx",
         "personal_data": true,
         "patient_identifier": false
        },
        {
         "field": "dob",
         "type": "string (date)",
         "meaning": "Date of birth, day first (per documentation)",
         "format": "DD/MM/YYYY",
         "personal_data": true,
         "patient_identifier": false
        },
        {
         "field": "sex",
         "type": "string (code)",
         "meaning": "Administrative sex",
         "format": "code: F/M/U",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "mobile",
         "type": "string",
         "meaning": "Mobile phone number",
         "format": "999 999 9999",
         "personal_data": true,
         "patient_identifier": false
        },
        {
         "field": "county",
         "type": "string",
         "meaning": "County of residence",
         "format": "free text",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "registered_on",
         "type": "string (date)",
         "meaning": "Date the patient registered",
         "format": "YYYY-MM-DD",
         "personal_data": false,
         "patient_identifier": false
        }
       ]
      }
     ]
    },
    {
     "system_id": "hospital_db",
     "summary": "Legacy appointments database with no documentation. TBL_APPT holds appointments; LKP_DEPT decodes department codes.",
     "access_method": "SQL (SQLite) queries against tables TBL_APPT and LKP_DEPT",
     "entities": [
      {
       "entity": "TBL_APPT",
       "fields": [
        {
         "field": "APPT_NO",
         "type": "integer",
         "meaning": "Appointment number (primary key)",
         "format": "4-digit integer",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "PT_ID",
         "type": "text",
         "meaning": "Patient identifier; same format as patient_id in the registration API",
         "format": "P + 3 digits",
         "personal_data": false,
         "patient_identifier": true
        },
        {
         "field": "APPT_DT",
         "type": "text (date-time)",
         "meaning": "Appointment date and time",
         "format": "YYYYMMDD HHMM",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "DEPT_CD",
         "type": "text (code)",
         "meaning": "Department code, decoded via LKP_DEPT",
         "format": "4-letter code, e.g. CARD",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "STAT",
         "type": "text (code)",
         "meaning": "Appointment status. B/A/C seen in samples; likely Booked/Attended/Cancelled (uncertain: no documentation)",
         "format": "code: single letter",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "CLIN_ID",
         "type": "text",
         "meaning": "Clinician identifier (uncertain: no clinician table found)",
         "format": "C + 2 digits",
         "personal_data": false,
         "patient_identifier": false
        }
       ]
      },
      {
       "entity": "LKP_DEPT",
       "fields": [
        {
         "field": "DEPT_CD",
         "type": "text",
         "meaning": "Department code (primary key)",
         "format": "4-letter code",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "DEPT_DESC",
         "type": "text",
         "meaning": "Department name",
         "format": "free text, e.g. Cardiology",
         "personal_data": false,
         "patient_identifier": false
        }
       ]
      }
     ]
    },
    {
     "system_id": "lab_csv",
     "summary": "Nightly laboratory export, one row per test result, LOINC-coded.",
     "access_method": "File ingest: CSV with header row",
     "entities": [
      {
       "entity": "lab_results.csv",
       "fields": [
        {
         "field": "patient_ref",
         "type": "string",
         "meaning": "Patient identifier; same format as patient_id",
         "format": "P + 3 digits",
         "personal_data": false,
         "patient_identifier": true
        },
        {
         "field": "sample_date",
         "type": "string (date)",
         "meaning": "Date the sample was collected",
         "format": "DD-MM-YYYY",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "test_code",
         "type": "string (code)",
         "meaning": "LOINC test code",
         "format": "digits-hyphen-check digit, e.g. 718-7",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "test_name",
         "type": "string",
         "meaning": "Test name",
         "format": "free text",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "result",
         "type": "string (number)",
         "meaning": "Result value, stored as text",
         "format": "decimal number",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "units",
         "type": "string",
         "meaning": "Unit of the result",
         "format": "e.g. g/dL, mmol/L, %",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "ref_range",
         "type": "string",
         "meaning": "Normal reference range",
         "format": "low-high, e.g. 12.0-15.5",
         "personal_data": false,
         "patient_identifier": false
        }
       ]
      }
     ]
    }
   ],
   "relationships": [
    {
     "from": "hospital_db.TBL_APPT.PT_ID",
     "to": "patient_api./patients.patient_id",
     "meaning": "Each appointment belongs to a registered patient"
    },
    {
     "from": "lab_csv.lab_results.csv.patient_ref",
     "to": "patient_api./patients.patient_id",
     "meaning": "Each lab result belongs to a registered patient"
    },
    {
     "from": "hospital_db.TBL_APPT.DEPT_CD",
     "to": "hospital_db.LKP_DEPT.DEPT_CD",
     "meaning": "Lookup table decodes department codes into names"
    }
   ],
   "data_quality_risks": [
    "Three different date formats: DD/MM/YYYY (API), YYYYMMDD HHMM (database), DD-MM-YYYY (CSV).",
    "Status codes in TBL_APPT are undocumented single letters; their meaning must be confirmed by a person.",
    "Lab results are stored as text, so non-numeric values (e.g. 'pending') may occur.",
    "The patient identifier may be missing in lab rows or refer to patients who are not registered.",
    "Names contain accents and apostrophes, so the file must be read as UTF-8."
   ]
  },
  "design": {
   "connections": [
    {
     "system_id": "patient_api",
     "pattern": "REST API pull",
     "how": "GET /patients, then validate and convert each record",
     "reason": "The system already exposes a JSON API"
    },
    {
     "system_id": "hospital_db",
     "pattern": "SQL batch query",
     "how": "SELECT from TBL_APPT LEFT JOIN LKP_DEPT on DEPT_CD",
     "reason": "No API exists; direct read-only query of the legacy database"
    },
    {
     "system_id": "lab_csv",
     "pattern": "File ingest",
     "how": "Read the nightly CSV export with a CSV reader (UTF-8)",
     "reason": "The lab system only produces file exports"
    }
   ],
   "mappings": [
    {
     "system_id": "patient_api",
     "source": "/patients.patient_id",
     "target": "Patient.id",
     "transform": "copy as-is after checking it is non-empty"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.full_name",
     "target": "Patient.name[0].family",
     "transform": "last word of full_name"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.full_name",
     "target": "Patient.name[0].given",
     "transform": "all words except the last, as a list"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.dob",
     "target": "Patient.birthDate",
     "transform": "DD/MM/YYYY -> YYYY-MM-DD"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.sex",
     "target": "Patient.gender",
     "transform": "F->female, M->male, O->other, U->unknown"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_APPT.APPT_NO",
     "target": "Appointment.id",
     "transform": "'APPT-' + APPT_NO"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_APPT.STAT",
     "target": "Appointment.status",
     "transform": "B->booked, A->fulfilled, C->cancelled, N->noshow (confirmed by reviewer)"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_APPT.APPT_DT",
     "target": "Appointment.start",
     "transform": "YYYYMMDD HHMM -> YYYY-MM-DDTHH:MM:SS"
    },
    {
     "system_id": "hospital_db",
     "source": "LKP_DEPT.DEPT_DESC (via TBL_APPT.DEPT_CD)",
     "target": "Appointment.serviceType[0].text",
     "transform": "join on DEPT_CD and copy the department name"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_APPT.PT_ID",
     "target": "Appointment.participant[0].actor.reference",
     "transform": "'Patient/' + PT_ID"
    },
    {
     "system_id": "lab_csv",
     "source": "patient_ref",
     "target": "Observation.subject.reference",
     "transform": "'Patient/' + patient_ref"
    },
    {
     "system_id": "lab_csv",
     "source": "sample_date",
     "target": "Observation.effectiveDateTime",
     "transform": "DD-MM-YYYY -> YYYY-MM-DD"
    },
    {
     "system_id": "lab_csv",
     "source": "test_code",
     "target": "Observation.code.coding[0].code",
     "transform": "copy; system = http://loinc.org"
    },
    {
     "system_id": "lab_csv",
     "source": "test_name",
     "target": "Observation.code.text",
     "transform": "copy"
    },
    {
     "system_id": "lab_csv",
     "source": "result",
     "target": "Observation.valueQuantity.value",
     "transform": "text -> decimal number"
    },
    {
     "system_id": "lab_csv",
     "source": "units",
     "target": "Observation.valueQuantity.unit",
     "transform": "copy"
    },
    {
     "system_id": "lab_csv",
     "source": "ref_range",
     "target": "Observation.referenceRange[0].text",
     "transform": "copy"
    },
    {
     "system_id": "lab_csv",
     "source": "(row number)",
     "target": "Observation.id",
     "transform": "'LAB-' + 4-digit row number"
    }
   ],
   "not_mapped": [
    {
     "system_id": "patient_api",
     "source": "/patients.mobile",
     "reason": "Personal contact data, not needed for the clinical record"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.county",
     "reason": "No target field agreed"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.registered_on",
     "reason": "Administrative, no target field agreed"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_APPT.CLIN_ID",
     "reason": "No clinician directory to resolve it; out of scope"
    }
   ],
   "validation_rules": [
    "Reject records with a missing or malformed patient identifier.",
    "Reject records whose dates cannot be parsed in the documented format.",
    "Reject appointment status codes outside B/A/C/N.",
    "Reject lab results that are not numeric (e.g. 'pending') and report them.",
    "Never write personal values into logs; report row number and field name only."
   ]
  },
  "generate_patient_api_attempt1": {
   "code": "import re\nfrom datetime import datetime\n\nGENDER_CODES = {\"F\": \"female\", \"M\": \"male\", \"O\": \"other\", \"U\": \"unknown\"}\n\ndef extract_patient_api(source, log):\n    \"\"\"Patient Registration API (REST) -> FHIR Patient records.\n    Rejected records are reported through log() with row number and field name only.\"\"\"\n    records = []\n    for row_number, row in enumerate(source(\"/patients\"), start=1):\n        try:\n            if not isinstance(row, dict):\n                raise ValueError(\"record\")\n            patient_id = str(row.get(\"patient_id\") or \"\").strip()\n            if not re.fullmatch(r\"[A-Za-z0-9\\-]+\", patient_id):\n                raise ValueError(\"patient_id\")\n            full_name = str(row.get(\"full_name\") or \"\").strip()\n            name_parts = full_name.split()\n            if len(name_parts) < 2 or any(ch.isdigit() or ch == \"?\" for ch in full_name):\n                raise ValueError(\"full_name\")\n            try:\n                birth_date = datetime.strptime(str(row.get(\"dob\") or \"\").strip(), \"%d/%m/%Y\").date().isoformat()\n            except ValueError:\n                raise ValueError(\"dob\")\n            gender = GENDER_CODES.get(str(row.get(\"sex\") or \"\").strip().upper())\n            if gender is None:\n                raise ValueError(\"sex\")\n        except ValueError as problem:\n            log(f\"row {row_number}: rejected - {problem} missing or invalid\")\n            continue\n        records.append({\"resourceType\": \"Patient\", \"id\": patient_id,\n                        \"name\": [{\"family\": name_parts[-1], \"given\": name_parts[:-1]}],\n                        \"gender\": gender, \"birthDate\": birth_date})\n    return records\n",
   "explanation": "Pulls all patients from GET /patients, splits the full name into given and family names, converts the day-first date of birth to ISO and maps sex codes to FHIR gender. Records with a bad ID, name, date or code are rejected and logged by row number.",
   "suggested_tests": [
    "A patient with dob 29/02/1990 (invalid date) is rejected.",
    "A patient with sex 'X' is rejected.",
    "A name with three words keeps the first two as given names.",
    "An empty API response returns an empty list."
   ]
  },
  "generate_hospital_db_attempt1": {
   "code": "import re\nfrom datetime import datetime\n\nSTATUS_CODES = {\"B\": \"booked\", \"A\": \"fulfilled\", \"C\": \"cancelled\", \"N\": \"noshow\"}\n\ndef extract_hospital_db(source, log):\n    \"\"\"Hospital database (SQL) -> FHIR Appointment records.\n    Joins TBL_APPT to LKP_DEPT to turn department codes into names.\n    Rejected records are reported through log() with row number and field name only.\"\"\"\n    records = []\n    query = \"\"\"SELECT a.APPT_NO, a.PT_ID, a.APPT_DT, a.STAT, d.DEPT_DESC\n               FROM TBL_APPT a LEFT JOIN LKP_DEPT d ON d.DEPT_CD = a.DEPT_CD\n               ORDER BY a.APPT_NO\"\"\"\n    for row_number, (appt_no, pt_id, appt_dt, stat, dept_desc) in enumerate(source.execute(query), start=1):\n        try:\n            if appt_no is None:\n                raise ValueError(\"APPT_NO\")\n            patient_id = str(pt_id or \"\").strip()\n            if not re.fullmatch(r\"[A-Za-z0-9\\-]+\", patient_id):\n                raise ValueError(\"PT_ID\")\n            try:\n                start = datetime.strptime(str(appt_dt or \"\").strip(), \"%Y%m%d %H%M\").isoformat()\n            except ValueError:\n                raise ValueError(\"APPT_DT\")\n            status = STATUS_CODES.get(str(stat or \"\").strip().upper())\n            if status is None:\n                raise ValueError(\"STAT\")\n        except ValueError as problem:\n            log(f\"row {row_number}: rejected - {problem} missing or invalid\")\n            continue\n        record = {\"resourceType\": \"Appointment\", \"id\": f\"APPT-{appt_no}\", \"status\": status, \"start\": start,\n                  \"participant\": [{\"actor\": {\"reference\": f\"Patient/{patient_id}\"}, \"status\": \"accepted\"}]}\n        if dept_desc:\n            record[\"serviceType\"] = [{\"text\": dept_desc}]\n        records.append(record)\n    return records\n",
   "explanation": "Runs one read-only query joining appointments to the department lookup, converts YYYYMMDD HHMM to ISO date-time and translates the status letters. Rows with a missing patient, bad date or unknown status are rejected and logged.",
   "suggested_tests": [
    "An appointment with an unknown department code still loads, without a serviceType.",
    "APPT_DT '20261340 2500' is rejected.",
    "Status 'X' is rejected.",
    "PT_ID with spaces around it is trimmed."
   ]
  },
  "generate_lab_csv_attempt1": {
   "code": "import csv\nimport re\nfrom datetime import datetime\n\ndef extract_lab_csv(source, log):\n    \"\"\"Laboratory system CSV export -> FHIR Observation records (LOINC-coded lab results).\n    Rejected records are reported through log() with row number and field name only.\"\"\"\n    records = []\n    with open(source, newline=\"\", encoding=\"utf-8-sig\") as handle:\n        for row_number, row in enumerate(csv.DictReader(handle), start=1):\n            try:\n                patient_id = (row.get(\"patient_ref\") or \"\").strip()\n                if not re.fullmatch(r\"[A-Za-z0-9\\-]+\", patient_id):\n                    raise ValueError(\"patient_ref\")\n                try:\n                    effective = datetime.strptime((row.get(\"sample_date\") or \"\").strip(), \"%d-%m-%Y\").date().isoformat()\n                except ValueError:\n                    raise ValueError(\"sample_date\")\n                code = (row.get(\"test_code\") or \"\").strip()\n                if not re.fullmatch(r\"\\d{1,7}-\\d\", code):\n                    raise ValueError(\"test_code\")\n                try:\n                    value = float((row.get(\"result\") or \"\").strip())\n                except ValueError:\n                    raise ValueError(\"result\")\n                unit = (row.get(\"units\") or \"\").strip()\n                if not unit:\n                    raise ValueError(\"units\")\n            except ValueError as problem:\n                log(f\"row {row_number}: rejected - {problem} missing or invalid\")\n                continue\n            test_name = (row.get(\"test_name\") or \"\").strip()\n            record = {\"resourceType\": \"Observation\", \"id\": f\"LAB-{row_number:04d}\", \"status\": \"final\",\n                      \"code\": {\"coding\": [{\"system\": \"http://loinc.org\", \"code\": code, \"display\": test_name}], \"text\": test_name},\n                      \"subject\": {\"reference\": f\"Patient/{patient_id}\"},\n                      \"effectiveDateTime\": effective,\n                      \"valueQuantity\": {\"value\": value, \"unit\": unit}}\n            ref_range = (row.get(\"ref_range\") or \"\").strip()\n            if ref_range:\n                record[\"referenceRange\"] = [{\"text\": ref_range}]\n            records.append(record)\n    return records\n",
   "explanation": "Reads the CSV export, converts DD-MM-YYYY to ISO dates, checks the LOINC code format and turns result text into a number. Rows with no patient, a bad date or a non-numeric result such as 'pending' are rejected and logged.",
   "suggested_tests": [
    "A result of 'pending' is rejected and logged.",
    "A row with no patient_ref is rejected.",
    "A result '5' is stored as the number 5.0.",
    "A file with only a header returns an empty list."
   ]
  }
 },
 "team": {
  "discover": {
   "systems": [
    {
     "system_id": "patient_api",
     "summary": "Patient registration system: one JSON record per registered patient with identity, demographics and address.",
     "access_method": "REST API: GET /patients (all) and GET /patients/{patient_id} (one)",
     "entities": [
      {
       "entity": "/patients",
       "fields": [
        {
         "field": "patient_id",
         "type": "string (UUID)",
         "meaning": "Hospital-wide patient identifier",
         "format": "8-4-4-4-12 hex UUID",
         "personal_data": false,
         "patient_identifier": true
        },
        {
         "field": "first_name",
         "type": "string",
         "meaning": "Given name",
         "format": "Xxxxx",
         "personal_data": true,
         "patient_identifier": false
        },
        {
         "field": "last_name",
         "type": "string",
         "meaning": "Family name",
         "format": "Xxxxx",
         "personal_data": true,
         "patient_identifier": false
        },
        {
         "field": "date_of_birth",
         "type": "string (date)",
         "meaning": "Date of birth, day first (per system notes)",
         "format": "DD/MM/YYYY",
         "personal_data": true,
         "patient_identifier": false
        },
        {
         "field": "sex",
         "type": "string (code)",
         "meaning": "Administrative sex",
         "format": "code: F/M",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "marital_status",
         "type": "string (code) or null",
         "meaning": "Marital status (uncertain codes, e.g. M/S)",
         "format": "single letter or null",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "address",
         "type": "string",
         "meaning": "Street address",
         "format": "free text",
         "personal_data": true,
         "patient_identifier": false
        },
        {
         "field": "city",
         "type": "string",
         "meaning": "City",
         "format": "free text",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "county",
         "type": "string",
         "meaning": "County",
         "format": "free text",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "postcode",
         "type": "string",
         "meaning": "Postal code",
         "format": "99999",
         "personal_data": true,
         "patient_identifier": false
        }
       ]
      }
     ]
    },
    {
     "system_id": "hospital_db",
     "summary": "Legacy visits database: every kind of visit (clinic, wellness, urgent care, emergency, admissions) in TBL_VISIT; LKP_VISIT_TYPE decodes visit types.",
     "access_method": "Read-only SQL queries on TBL_VISIT joined to LKP_VISIT_TYPE",
     "entities": [
      {
       "entity": "TBL_VISIT",
       "fields": [
        {
         "field": "VISIT_NO",
         "type": "integer",
         "meaning": "Visit number (primary key)",
         "format": "6-digit integer",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "PT_REF",
         "type": "text (UUID)",
         "meaning": "Patient identifier; same format as patient_id in the API",
         "format": "UUID",
         "personal_data": false,
         "patient_identifier": true
        },
        {
         "field": "VISIT_START",
         "type": "text (date-time)",
         "meaning": "Visit start",
         "format": "YYYYMMDD HHMM",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "VISIT_END",
         "type": "text (date-time)",
         "meaning": "Visit end",
         "format": "YYYYMMDD HHMM",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "VISIT_TYPE",
         "type": "text (code)",
         "meaning": "Visit type, decoded via LKP_VISIT_TYPE",
         "format": "3-letter code, e.g. AMB, EMR",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "SCT_CD",
         "type": "text (code)",
         "meaning": "SNOMED CT code of the visit (uncertain: inferred from values and name)",
         "format": "numeric code",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "VISIT_DESC",
         "type": "text",
         "meaning": "Description of the visit",
         "format": "free text",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "RSN_DESC",
         "type": "text",
         "meaning": "Reason for the visit, often empty",
         "format": "free text or null",
         "personal_data": false,
         "patient_identifier": false
        }
       ]
      },
      {
       "entity": "LKP_VISIT_TYPE",
       "fields": [
        {
         "field": "TYPE_CD",
         "type": "text",
         "meaning": "Visit type code (primary key)",
         "format": "3 letters",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "TYPE_DESC",
         "type": "text",
         "meaning": "Visit type in words",
         "format": "free text",
         "personal_data": false,
         "patient_identifier": false
        }
       ]
      }
     ]
    },
    {
     "system_id": "lab_csv",
     "summary": "Nightly laboratory export: one row per numeric lab result, LOINC-coded.",
     "access_method": "File ingest: UTF-8 CSV with a header row",
     "entities": [
      {
       "entity": "lab_results.csv",
       "fields": [
        {
         "field": "patient_ref",
         "type": "string (UUID)",
         "meaning": "Patient identifier; same format as patient_id",
         "format": "UUID",
         "personal_data": false,
         "patient_identifier": true
        },
        {
         "field": "collected_at",
         "type": "string (date-time)",
         "meaning": "Sample collection time, day first (per system notes)",
         "format": "DD/MM/YYYY HH:MM",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "loinc_code",
         "type": "string (code)",
         "meaning": "LOINC test code",
         "format": "digits-hyphen-check digit, e.g. 4544-3",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "test_description",
         "type": "string",
         "meaning": "Test name",
         "format": "free text",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "result_value",
         "type": "string (number)",
         "meaning": "Result value, stored as text",
         "format": "decimal number",
         "personal_data": false,
         "patient_identifier": false
        },
        {
         "field": "result_unit",
         "type": "string",
         "meaning": "Unit of the result (UCUM-style)",
         "format": "e.g. %, fL, mg/dL",
         "personal_data": false,
         "patient_identifier": false
        }
       ]
      }
     ]
    }
   ],
   "relationships": [
    {
     "from": "hospital_db.TBL_VISIT.PT_REF",
     "to": "patient_api./patients.patient_id",
     "meaning": "Each visit belongs to a registered patient"
    },
    {
     "from": "lab_csv.lab_results.csv.patient_ref",
     "to": "patient_api./patients.patient_id",
     "meaning": "Each lab result belongs to a registered patient"
    },
    {
     "from": "hospital_db.TBL_VISIT.VISIT_TYPE",
     "to": "hospital_db.LKP_VISIT_TYPE.TYPE_CD",
     "meaning": "Lookup table decodes visit-type codes"
    }
   ],
   "data_quality_risks": [
    "Three date styles: DD/MM/YYYY (API), YYYYMMDD HHMM (database), DD/MM/YYYY HH:MM (CSV); day-first dates are easy to misread as month-first.",
    "Lab results are stored as text, so non-numeric values such as 'pending' may occur.",
    "Patient identifiers may be missing, or point to patients who are not registered.",
    "The visits table mixes past visits of every type; none are future bookings.",
    "Addresses and postcodes are personal data and should not be copied into the clinical record."
   ]
  },
  "design": {
   "connections": [
    {
     "system_id": "patient_api",
     "pattern": "REST API pull",
     "how": "GET /patients, validate and convert each record",
     "reason": "The system already exposes a JSON API"
    },
    {
     "system_id": "hospital_db",
     "pattern": "SQL batch query",
     "how": "SELECT from TBL_VISIT LEFT JOIN LKP_VISIT_TYPE on VISIT_TYPE = TYPE_CD",
     "reason": "No API exists; read-only query of the legacy database"
    },
    {
     "system_id": "lab_csv",
     "pattern": "File ingest",
     "how": "Read the nightly CSV export with a CSV reader (UTF-8)",
     "reason": "The lab system only produces file exports"
    }
   ],
   "mappings": [
    {
     "system_id": "patient_api",
     "source": "/patients.patient_id",
     "target": "Patient.id",
     "transform": "copy as-is after checking it is a valid identifier"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.last_name",
     "target": "Patient.name[0].family",
     "transform": "copy"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.first_name",
     "target": "Patient.name[0].given",
     "transform": "split on spaces into a list"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.date_of_birth",
     "target": "Patient.birthDate",
     "transform": "DD/MM/YYYY -> YYYY-MM-DD"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.sex",
     "target": "Patient.gender",
     "transform": "F->female, M->male, O->other, U->unknown"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_VISIT.VISIT_NO",
     "target": "Encounter.id",
     "transform": "'ENC-' + VISIT_NO"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_VISIT.VISIT_END",
     "target": "Encounter.status",
     "transform": "'finished' when VISIT_END is present, otherwise 'in-progress'"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_VISIT.VISIT_START",
     "target": "Encounter.period.start",
     "transform": "YYYYMMDD HHMM -> YYYY-MM-DDTHH:MM:SS"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_VISIT.VISIT_END",
     "target": "Encounter.period.end",
     "transform": "YYYYMMDD HHMM -> YYYY-MM-DDTHH:MM:SS (optional)"
    },
    {
     "system_id": "hospital_db",
     "source": "LKP_VISIT_TYPE.TYPE_DESC (via TBL_VISIT.VISIT_TYPE)",
     "target": "Encounter.serviceType.text",
     "transform": "join on the visit-type code and copy the description"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_VISIT.PT_REF",
     "target": "Encounter.subject.reference",
     "transform": "'Patient/' + PT_REF"
    },
    {
     "system_id": "lab_csv",
     "source": "patient_ref",
     "target": "Observation.subject.reference",
     "transform": "'Patient/' + patient_ref"
    },
    {
     "system_id": "lab_csv",
     "source": "collected_at",
     "target": "Observation.effectiveDateTime",
     "transform": "DD/MM/YYYY HH:MM -> YYYY-MM-DDTHH:MM:SS"
    },
    {
     "system_id": "lab_csv",
     "source": "loinc_code",
     "target": "Observation.code.coding[0].code",
     "transform": "copy; system = http://loinc.org"
    },
    {
     "system_id": "lab_csv",
     "source": "test_description",
     "target": "Observation.code.text",
     "transform": "copy"
    },
    {
     "system_id": "lab_csv",
     "source": "result_value",
     "target": "Observation.valueQuantity.value",
     "transform": "text -> decimal number"
    },
    {
     "system_id": "lab_csv",
     "source": "result_unit",
     "target": "Observation.valueQuantity.unit",
     "transform": "copy"
    },
    {
     "system_id": "lab_csv",
     "source": "(row number)",
     "target": "Observation.id",
     "transform": "'LAB-' + 5-digit row number"
    }
   ],
   "not_mapped": [
    {
     "system_id": "patient_api",
     "source": "/patients.address, city, county, postcode",
     "reason": "Personal contact data, not needed for the clinical record"
    },
    {
     "system_id": "patient_api",
     "source": "/patients.marital_status",
     "reason": "No target field agreed"
    },
    {
     "system_id": "hospital_db",
     "source": "TBL_VISIT.SCT_CD, VISIT_DESC, RSN_DESC",
     "reason": "No target field agreed (a candidate for Encounter.type and reasonCode later)"
    }
   ],
   "validation_rules": [
    "Reject records with a missing or malformed patient identifier.",
    "Reject records whose dates cannot be parsed in the documented day-first or YYYYMMDD format.",
    "Reject lab results that are not numeric (e.g. 'pending') or have no unit, and report them.",
    "Flag (do not drop) records that refer to a patient missing from the registration system.",
    "Never write personal values into logs; report row number and field name only."
   ]
  },
  "generate_patient_api_attempt1": {
   "code": "import re\nfrom datetime import datetime\n\nGENDER_CODES = {\"F\": \"female\", \"M\": \"male\", \"O\": \"other\", \"U\": \"unknown\"}\n\ndef extract_patient_api(source, log):\n    \"\"\"Patient Registration API (REST) -> FHIR Patient records.\n    Rejected records are reported through log() with row number and field name only.\"\"\"\n    records = []\n    for row_number, row in enumerate(source(\"/patients\"), start=1):\n        try:\n            if not isinstance(row, dict):\n                raise ValueError(\"record\")\n            patient_id = str(row.get(\"patient_id\") or \"\").strip()\n            if not re.fullmatch(r\"[A-Za-z0-9\\-]+\", patient_id):\n                raise ValueError(\"patient_id\")\n            first_name = str(row.get(\"first_name\") or \"\").strip()\n            last_name = str(row.get(\"last_name\") or \"\").strip()\n            if not first_name or any(ch.isdigit() or ch == \"?\" for ch in first_name):\n                raise ValueError(\"first_name\")\n            if not last_name or any(ch.isdigit() or ch == \"?\" for ch in last_name):\n                raise ValueError(\"last_name\")\n            try:\n                birth_date = datetime.strptime(str(row.get(\"date_of_birth\") or \"\").strip(), \"%d/%m/%Y\").date().isoformat()\n            except ValueError:\n                raise ValueError(\"date_of_birth\")\n            gender = GENDER_CODES.get(str(row.get(\"sex\") or \"\").strip().upper())\n            if gender is None:\n                raise ValueError(\"sex\")\n        except ValueError as problem:\n            log(f\"row {row_number}: rejected - {problem} missing or invalid\")\n            continue\n        records.append({\"resourceType\": \"Patient\", \"id\": patient_id,\n                        \"name\": [{\"family\": last_name, \"given\": first_name.split()}],\n                        \"gender\": gender, \"birthDate\": birth_date})\n    return records\n",
   "explanation": "Pulls every patient from GET /patients, converts the day-first date of birth to ISO and the sex code to FHIR gender, and keeps only the name fields. Address data is deliberately not copied. Invalid records are rejected and logged by row number.",
   "suggested_tests": [
    "A patient whose date_of_birth is 31/02/1990 is rejected.",
    "A patient with sex 'X' is rejected.",
    "A first name with two words becomes two given names.",
    "Address and postcode never appear in the output."
   ]
  },
  "generate_hospital_db_attempt1": {
   "code": "import re\nfrom datetime import datetime\n\ndef extract_hospital_db(source, log):\n    \"\"\"Hospital database (SQL) -> FHIR Encounter records (past visits of every type).\n    Joins TBL_VISIT to LKP_VISIT_TYPE to turn visit-type codes into words.\n    Rejected records are reported through log() with row number and field name only.\"\"\"\n    records = []\n    query = \"\"\"SELECT v.VISIT_NO, v.PT_REF, v.VISIT_START, v.VISIT_END, t.TYPE_DESC\n               FROM TBL_VISIT v LEFT JOIN LKP_VISIT_TYPE t ON t.TYPE_CD = v.VISIT_TYPE\n               ORDER BY v.VISIT_NO\"\"\"\n    for row_number, (visit_no, pt_ref, visit_start, visit_end, type_desc) in enumerate(source.execute(query), start=1):\n        try:\n            if visit_no is None:\n                raise ValueError(\"VISIT_NO\")\n            patient_id = str(pt_ref or \"\").strip()\n            if not re.fullmatch(r\"[A-Za-z0-9\\-]+\", patient_id):\n                raise ValueError(\"PT_REF\")\n            try:\n                start = datetime.strptime(str(visit_start or \"\").strip(), \"%Y%m%d %H%M\").isoformat()\n            except ValueError:\n                raise ValueError(\"VISIT_START\")\n            end = None\n            if visit_end:\n                try:\n                    end = datetime.strptime(str(visit_end).strip(), \"%Y%m%d %H%M\").isoformat()\n                except ValueError:\n                    raise ValueError(\"VISIT_END\")\n        except ValueError as problem:\n            log(f\"row {row_number}: rejected - {problem} missing or invalid\")\n            continue\n        period = {\"start\": start}\n        if end:\n            period[\"end\"] = end\n        record = {\"resourceType\": \"Encounter\", \"id\": f\"ENC-{visit_no}\",\n                  \"status\": \"finished\" if end else \"in-progress\", \"period\": period,\n                  \"subject\": {\"reference\": f\"Patient/{patient_id}\"}}\n        if type_desc:\n            record[\"serviceType\"] = {\"text\": type_desc}\n        records.append(record)\n    return records\n",
   "explanation": "Runs one read-only query joining visits to the visit-type lookup, converts YYYYMMDD HHMM to ISO date-times, and marks visits with an end time as finished. Visits with no patient or an unreadable date are rejected and logged.",
   "suggested_tests": [
    "A visit with VISIT_START '20261340 2500' is rejected.",
    "A visit with no VISIT_END becomes status in-progress.",
    "An unknown VISIT_TYPE still loads, without serviceType.",
    "PT_REF with surrounding spaces is trimmed."
   ]
  },
  "generate_lab_csv_attempt1": {
   "code": "import csv\nimport re\nfrom datetime import datetime\n\ndef extract_lab_csv(source, log):\n    \"\"\"Laboratory system CSV export -> FHIR Observation records (LOINC-coded numeric lab results).\n    Rejected records are reported through log() with row number and field name only.\"\"\"\n    records = []\n    with open(source, newline=\"\", encoding=\"utf-8-sig\") as handle:\n        for row_number, row in enumerate(csv.DictReader(handle), start=1):\n            try:\n                patient_id = (row.get(\"patient_ref\") or \"\").strip()\n                if not re.fullmatch(r\"[A-Za-z0-9\\-]+\", patient_id):\n                    raise ValueError(\"patient_ref\")\n                try:\n                    effective = datetime.strptime((row.get(\"collected_at\") or \"\").strip(), \"%d/%m/%Y %H:%M\").isoformat()\n                except ValueError:\n                    raise ValueError(\"collected_at\")\n                code = (row.get(\"loinc_code\") or \"\").strip()\n                if not re.fullmatch(r\"\\d{1,7}-\\d\", code):\n                    raise ValueError(\"loinc_code\")\n                try:\n                    value = float((row.get(\"result_value\") or \"\").strip())\n                except ValueError:\n                    raise ValueError(\"result_value\")\n                unit = (row.get(\"result_unit\") or \"\").strip()\n                if not unit:\n                    raise ValueError(\"result_unit\")\n            except ValueError as problem:\n                log(f\"row {row_number}: rejected - {problem} missing or invalid\")\n                continue\n            description = (row.get(\"test_description\") or \"\").strip()\n            records.append({\"resourceType\": \"Observation\", \"id\": f\"LAB-{row_number:05d}\", \"status\": \"final\",\n                            \"code\": {\"coding\": [{\"system\": \"http://loinc.org\", \"code\": code, \"display\": description}], \"text\": description},\n                            \"subject\": {\"reference\": f\"Patient/{patient_id}\"},\n                            \"effectiveDateTime\": effective,\n                            \"valueQuantity\": {\"value\": value, \"unit\": unit}})\n    return records\n",
   "explanation": "Reads the CSV export, converts DD/MM/YYYY HH:MM to ISO date-times, checks the LOINC code format and converts the result to a number. Rows with no patient, a bad date or a non-numeric result such as 'pending' are rejected and logged.",
   "suggested_tests": [
    "A result_value of 'pending' is rejected and logged.",
    "A row with an empty patient_ref is rejected.",
    "collected_at 13/11/2026 09:05 becomes 2026-11-13T09:05:00.",
    "A row with an empty result_unit is rejected."
   ]
  }
 }
}''')
REPLAY_EXAMPLES = REPLAY_SETS.get(DATA_SOURCE, {})
print(f"{len(REPLAY_EXAMPLES)} backup outputs loaded.")

---
## Stage 1 · Discover
**What happens:** the agent's tools open each system itself: they call the API, read the database schema and sample rows, and read the CSV. Personal fields are **masked before anything is sent to the AI model**: the format stays visible, the content doesn't (`Aoife Murphy` → `Xxxxx Xxxxxx`). The agent then explains each system, every field, and how the systems link together.

**What the reviewer checks:** Are the field meanings right? Did it find the patient ID link across all three systems? Are the date formats right?

In [ ]:
#@title 6 · STAGE 1: Discover · the agent inspects the three systems { display-mode: "form" }
def inspect_patient_api():
    records = api_get("/patients")
    return {"system_id": "patient_api", "kind": "REST API", "documentation": DOCS.get("patient_api", ""),
            "calls_made": ["GET /patients"], "record_count": len(records),
            "sample_records_masked": [mask_record(r) for r in records[:5]]}

def inspect_hospital_db():
    connection = sqlite3.connect(DB_FILE)
    tables = []
    for name, create_sql in connection.execute("SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'").fetchall():
        cursor = connection.execute(f'SELECT * FROM "{name}" LIMIT 5')
        columns = [c[0] for c in cursor.description]
        rows = [mask_record(dict(zip(columns, row))) for row in cursor.fetchall()]
        count = connection.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
        tables.append({"table": name, "create_statement": create_sql, "row_count": count, "sample_rows_masked": rows})
    connection.close()
    return {"system_id": "hospital_db", "kind": "SQL database", "documentation": DOCS.get("hospital_db", ""), "tables": tables}

def inspect_lab_csv():
    with open(LAB_FILE, newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    return {"system_id": "lab_csv", "kind": "CSV file", "documentation": DOCS.get("lab_csv", ""),
            "columns": list(rows[0].keys()) if rows else [], "row_count": len(rows),
            "sample_rows_masked": [mask_record(r) for r in rows[:5]]}

evidence = {"patient_api": inspect_patient_api(), "hospital_db": inspect_hospital_db(), "lab_csv": inspect_lab_csv()}
if DOCS.get("shared"):
    evidence["shared_documentation"] = DOCS["shared"]
audit("discovery_tools_run", systems=SYSTEM_IDS)
print("🔎 The agent's tools opened all three systems. Personal fields were masked before being sent to the AI model.")

DISCOVER_PROMPT = """You are the DISCOVERY step of an AI integration agent for a hospital group.
Your tools collected the evidence below from three source systems you have never seen before: a REST API, a SQL database and a CSV file.
Personal fields were masked before you received them: letters became X/x and digits became 9, so you can see each value's FORMAT but not its content.

Your job:
1. For every system, explain what it holds and how it is accessed.
2. For every field: its type, its meaning (infer it from names, documentation and samples), its exact format
   (e.g. "DD/MM/YYYY", "YYYYMMDD HHMM", "code: B/A/C/N"), whether it is personal data, and whether it identifies the patient.
3. Work out how the systems link together (which fields hold the same patient identifier, which lookup tables decode which codes).
4. List the data-quality risks a developer must handle (formats, codes, missing values, units).
Only state what the evidence supports. If a meaning is unclear, say "uncertain" and why.

Reply with JSON only, in exactly this shape:
{"systems": [{"system_id": "patient_api | hospital_db | lab_csv", "summary": "...", "access_method": "...",
  "entities": [{"entity": "table, endpoint or file name", "fields": [{"field": "...", "type": "...", "meaning": "...", "format": "...", "personal_data": true, "patient_identifier": false}]}]}],
 "relationships": [{"from": "system.entity.field", "to": "system.entity.field", "meaning": "..."}],
 "data_quality_risks": ["..."]}

EVIDENCE:
""" + json.dumps(evidence, indent=1, default=str, ensure_ascii=False)

discovery, seconds = ask_agent("discover", DISCOVER_PROMPT)
with open(f"{OUT}/1_discovery.json", "w") as handle:
    json.dump(discovery, handle, indent=2, ensure_ascii=False)

for system in discovery.get("systems", []):
    heading(f"{SYSTEM_LABELS.get(system.get('system_id'), system.get('system_id'))}")
    print(system.get("summary", ""), "\nAccess:", system.get("access_method", ""))
    field_rows = [{"entity": entity.get("entity"), **field} for entity in system.get("entities", []) for field in entity.get("fields", [])]
    display(pd.DataFrame(field_rows))
heading("How the systems link together")
display(pd.DataFrame(discovery.get("relationships", [])))
heading("Data-quality risks the agent spotted")
for risk in discovery.get("data_quality_risks", []):
    print(" •", risk)
stage_finished("discover", agent_seconds=seconds)

In [ ]:
#@title ✅ Human review · Discover { display-mode: "form" }
#@markdown Read the agent's summary above. If it's right, type your name, tick APPROVED and run this cell. Add notes for the Design step if anything needs correcting.
REVIEWER = "Jewel"  #@param {type:"string"}
APPROVED = True  #@param {type:"boolean"}
NOTES = ""  #@param {type:"string"}
approve("discover", REVIEWER, APPROVED, NOTES)

---
## Stage 2 · Design
**What happens:** the agent takes the **approved** discovery summary and the agreed FHIR structure. It then decides how to connect to each system (API pull, SQL query, file ingest) and maps every source field to a FHIR field, describing each conversion exactly. It also produces an **OpenAPI spec** for the common API.

**What the reviewer checks:** every mapping and conversion. For example: is `dob` really day-first? Is status code `N` really "no-show"? If something is wrong, type it in `DESIGN_FEEDBACK` and re-run the cell (live mode).

In [ ]:
#@title 7 · STAGE 2: Design · map every field to FHIR and choose how to connect { display-mode: "form" }
#@markdown If the reviewer spots a wrong mapping, describe it here and re-run this cell (live mode), e.g. *"STAT code N means no-show"*.
DESIGN_FEEDBACK = ""  #@param {type:"string"}
require_approved("discover")

DESIGN_PROMPT = """You are the DESIGN step of an AI integration agent for a hospital group.
You receive the discovery summary that a human reviewer approved, and the target structure (FHIR-aligned) the team agreed.

Your job:
1. For each system choose the connection pattern (REST API pull, SQL batch query, or file ingest) and say why.
2. Map every source field that carries identity or clinical meaning to a target field. Write targets as "Resource.path"
   using the target paths exactly as written in TARGET (e.g. "Patient.birthDate", "Appointment.participant[0].actor.reference").
   Describe each transform precisely: input format -> output format, and code tables in full (e.g. "F->female, M->male").
3. List the source fields you deliberately do NOT map, with the reason (e.g. personal data with no target field).
4. List the validation rules the generated code must enforce.
Only use resources and fields that exist in TARGET. Use Encounter only for admissions or stays, not appointments.

Reply with JSON only, in exactly this shape:
{"connections": [{"system_id": "...", "pattern": "...", "how": "...", "reason": "..."}],
 "mappings": [{"system_id": "...", "source": "entity.field", "target": "Resource.path", "transform": "..."}],
 "not_mapped": [{"system_id": "...", "source": "...", "reason": "..."}],
 "validation_rules": ["..."]}

REVIEWER NOTES FROM DISCOVERY: """ + (STAGES["discover"].get("notes") or "none") + """
REVIEWER FEEDBACK ON A PREVIOUS DESIGN: """ + (DESIGN_FEEDBACK or "none") + """
TARGET: """ + json.dumps(TARGET_FOR_PROMPT, indent=1) + """
SHARED RULES: """ + json.dumps(SHARED_RULES, indent=1) + """
APPROVED DISCOVERY: """ + json.dumps(discovery, indent=1, ensure_ascii=False)

design, seconds = ask_agent("design", DESIGN_PROMPT)

def build_openapi(resources):
    """OpenAPI description of the common API, built from the approved design."""
    schemas = {}
    for resource in sorted(resources):
        spec = TARGET[resource]
        top_level = sorted({re.split(r"[.\[]", f)[0] for f in spec["fields"]})
        schemas[resource] = {"type": "object",
                             "required": ["resourceType"] + sorted({re.split(r"[.\[]", f)[0] for f in spec["required"]}),
                             "properties": {"resourceType": {"type": "string", "enum": [resource]},
                                            **{f: {"description": "; ".join(v for k, v in spec["fields"].items() if re.split(r"[.\[]", k)[0] == f)} for f in top_level}},
                             "example": spec["example"]}
    schemas["Bundle"] = {"type": "object", "properties": {"resourceType": {"type": "string", "enum": ["Bundle"]},
                         "type": {"type": "string"}, "total": {"type": "integer"},
                         "entry": {"type": "array", "items": {"type": "object", "properties": {"resource": {"oneOf": [{"$ref": f"#/components/schemas/{r}"} for r in sorted(resources)]}}}}}}
    paths = {"/Patient/{id}/$everything": {"get": {
        "summary": "Everything known about one patient, combined from all connected systems",
        "parameters": [{"name": "id", "in": "path", "required": True, "schema": {"type": "string"}}],
        "responses": {"200": {"description": "FHIR searchset Bundle", "content": {"application/fhir+json": {"schema": {"$ref": "#/components/schemas/Bundle"}}}},
                      "404": {"description": "Unknown patient"}}}}}
    for resource in sorted(resources):
        paths[f"/{resource}/{{id}}"] = {"get": {"summary": f"Read one {resource}",
            "parameters": [{"name": "id", "in": "path", "required": True, "schema": {"type": "string"}}],
            "responses": {"200": {"description": resource, "content": {"application/fhir+json": {"schema": {"$ref": f"#/components/schemas/{resource}"}}}},
                          "404": {"description": "Not found"}}}}
    return {"openapi": "3.0.3", "info": {"title": "Unified patient data API (FHIR-aligned)", "version": "0.1.0",
            "description": "Generated from the agent's approved design. Synthetic data only."},
            "paths": paths, "components": {"schemas": schemas}}

mapped_resources = {m.get("target", "").split(".")[0] for m in design.get("mappings", [])} & set(TARGET)
openapi_spec = build_openapi(mapped_resources)
with open(f"{OUT}/2_design.json", "w") as handle:
    json.dump(design, handle, indent=2, ensure_ascii=False)
pd.DataFrame(design.get("mappings", [])).to_csv(f"{OUT}/2_mappings.csv", index=False)
with open(f"{OUT}/2_openapi_common_api.json", "w") as handle:
    json.dump(openapi_spec, handle, indent=2)

heading("How the agent will connect to each system")
display(pd.DataFrame(design.get("connections", [])))
heading("Field mappings (source → FHIR)")
display(pd.DataFrame(design.get("mappings", [])))
heading("Fields deliberately not mapped")
display(pd.DataFrame(design.get("not_mapped", [])))
heading("Validation rules the code must enforce")
for rule in design.get("validation_rules", []):
    print(" •", rule)
print("\n📄 Common API spec (OpenAPI) written to outputs/2_openapi_common_api.json. Endpoints:", ", ".join(openapi_spec["paths"]))
stage_finished("design", agent_seconds=seconds, mappings=len(design.get("mappings", [])))

In [ ]:
#@title ✅ Human review · Design { display-mode: "form" }
#@markdown Check every mapping and conversion above. Approve only when you're happy the code can be generated from it.
REVIEWER = "Jewel"  #@param {type:"string"}
APPROVED = True  #@param {type:"boolean"}
NOTES = ""  #@param {type:"string"}
approve("design", REVIEWER, APPROVED, NOTES)

---
## Stage 3 · Generate, validate and self-heal
**What happens:** for each system the agent writes one integration function (an "adapter") from the **approved** mappings, all following the same template. Every adapter then goes through:
1. **A safety check before running:** no unapproved imports, no file writing, no network access, no `print`.
2. **The same automated checks for every adapter:** required fields filled, ISO dates, valid patient links, allowed codes, **broken records rejected instead of crashing**, and **no personal data in log messages**. On the placeholder data there are also spot checks against known correct values.
3. **Self-healing:** if a check fails, the failures go back to the agent, which fixes its own code and tries again (up to `MAX_FIX_ATTEMPTS`).

**What the reviewer checks:** that all checks pass, and a quick read of each adapter's code. Person 3 can also run the agent's suggested tests.

In [ ]:
#@title 8 · STAGE 3: Generate · the agent writes, tests and fixes the integration code { display-mode: "form" }
require_approved("design")

SOURCE_DESCRIPTIONS = {
    "patient_api": "a function api_get(path) that performs an HTTP GET on the Patient Registration REST API and returns the parsed JSON. Call source('/patients') to get the list of all patients.",
    "hospital_db": "an open sqlite3.Connection to the hospital database. Read with source.execute(sql). Join lookup tables to decode codes where the design says so.",
    "lab_csv": "a file path (string) to the laboratory CSV export. Read it with open(source, newline='', encoding='utf-8-sig') and csv.DictReader.",
}
TEMPLATE = '''import re
from datetime import datetime

def extract_<system_id>(source, log):
    """<System name> -> FHIR <Resource> records.
    Rejected records are reported through log() with row number and field name only."""
    records = []
    for row_number, row in enumerate(<read the rows from source>, start=1):
        try:
            <read and validate each field; on a problem: raise ValueError("<field name>")>
        except ValueError as problem:
            log(f"row {row_number}: rejected - {problem} missing or invalid")
            continue
        records.append({"resourceType": "<Resource>", "id": ..., ...})
    return records
'''
ALLOWED_IMPORTS = {"re", "datetime", "csv", "json", "sqlite3", "math"}
BLOCKED_NAMES = {"eval", "exec", "compile", "__import__", "globals", "locals", "vars", "getattr", "setattr", "delattr",
                 "input", "print", "breakpoint", "os", "sys", "subprocess", "shutil", "socket", "requests", "urllib", "pathlib"}

def contract_text(system_id, mappings):
    return ("FUNCTION CONTRACT\n"
            f"- Write exactly one function: def extract_{system_id}(source, log):\n"
            f"- `source` is {SOURCE_DESCRIPTIONS[system_id]}\n"
            "- Return a list of dicts, one per valid source record, shaped like the TARGET examples (include 'resourceType').\n"
            "- `log` is a function. For every record you reject call log('row <n>: rejected - <field name> missing or invalid').\n"
            "  NEVER put source values (names, dates of birth, phone numbers, IDs) into log messages. Only row numbers and field names.\n"
            "- Validate every record. If a required field is missing or cannot be converted (bad date, non-numeric value, unknown code),\n"
            "  reject that record with log(...) and continue. One bad record must never stop the run, and never output a half-filled record.\n"
            "- Patient IDs must be non-empty and contain only letters, digits and hyphens. Reject anything else.\n"
            "- Allowed imports: re, datetime, csv, json, sqlite3, math. No other imports, no file writing, no network, no print().\n"
            "- Start the function with a docstring saying which system it reads and which resources it returns.\n"
            "- Follow the SHARED TEMPLATE structure so every adapter looks the same.\n\n"
            "SHARED TEMPLATE:\n" + TEMPLATE + "\n"
            "SHARED RULES: " + json.dumps(SHARED_RULES) + "\n"
            "TARGET: " + json.dumps({k: v for k, v in TARGET_FOR_PROMPT.items() if k in {m.get('target', '').split('.')[0] for m in mappings}}, indent=1) + "\n"
            "APPROVED MAPPINGS FOR THIS SYSTEM: " + json.dumps(mappings, indent=1, ensure_ascii=False) + "\n"
            "APPROVED VALIDATION RULES: " + json.dumps(design.get("validation_rules", []), ensure_ascii=False) + "\n"
            "WHAT DISCOVERY FOUND ABOUT THIS SYSTEM (masked samples): " + json.dumps([s for s in discovery.get("systems", []) if s.get("system_id") == system_id], ensure_ascii=False) + "\n"
            "RAW STRUCTURE (masked samples): " + json.dumps(evidence[system_id], default=str, ensure_ascii=False) + "\n\n"
            'Reply with JSON only: {"code": "<the complete Python source>", "explanation": "<2-3 sentences>", "suggested_tests": ["<plain-English test case>", "..."]}')

def generation_prompt(system_id, mappings):
    return ("You are the GENERATE step of an AI integration agent for a hospital group. Write the integration code (an adapter) "
            f"for the system '{system_id}' ({SYSTEM_LABELS[system_id]}) that follows the human-approved design exactly.\n\n" + contract_text(system_id, mappings))

def heal_prompt(system_id, mappings, code, failures):
    return ("You are the GENERATE step of an AI integration agent. Your adapter for '" + system_id + "' failed automated checks. "
            "Fix the code so every check passes. Keep the same function name, template and approved mappings.\n\n"
            "FAILED CHECKS:\n" + "\n".join(f"- {f['test']}: {f['detail']}" for f in failures) + "\n\nYOUR PREVIOUS CODE:\n" + code + "\n\n" + contract_text(system_id, mappings))

def safety_check(code, system_id):
    """Static check of generated code before it is allowed to run."""
    try:
        tree = ast.parse(code)
    except SyntaxError as error:
        return [f"code does not parse: {error.msg} (line {error.lineno})"]
    problems = []
    for node in ast.walk(tree):
        modules = [a.name for a in node.names] if isinstance(node, ast.Import) else ([node.module or ""] if isinstance(node, ast.ImportFrom) else [])
        problems += [f"import of '{m}' is not allowed" for m in modules if m.split(".")[0] not in ALLOWED_IMPORTS]
        if isinstance(node, ast.Name) and node.id in BLOCKED_NAMES:
            problems.append(f"use of '{node.id}' is not allowed")
        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            problems.append("access to hidden '__' attributes is not allowed")
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "open":
            modes = [a.value for a in node.args[1:2] if isinstance(a, ast.Constant)]
            modes += [k.value.value for k in node.keywords if k.arg == "mode" and isinstance(k.value, ast.Constant)]
            if any(set(str(m)) & set("wax+") for m in modes):
                problems.append("writing files is not allowed")
    functions = [n.name for n in tree.body if isinstance(n, ast.FunctionDef)]
    if f"extract_{system_id}" not in functions:
        problems.append(f"function extract_{system_id}(source, log) is missing")
    return sorted(set(problems))

def run_adapter(function, source):
    messages = []
    try:
        records = function(source, messages.append)
        return records, messages, None
    except Exception as error:
        return None, messages, redact(f"{type(error).__name__}: {error}")[:300]

def make_bad_source(system_id):
    """A copy of the source with two broken records added: one empty, one full of garbage ('???')."""
    if system_id == "patient_api":
        good = api_get("/patients")
        template = good[0] if good else {"patient_id": None}
        broken = [{k: None for k in template}, {k: "???" for k in template}]
        def bad_api_get(path):
            return good + broken if path.rstrip("/") == "/patients" else api_get(path)
        return bad_api_get, None
    if system_id == "hospital_db":
        connection = sqlite3.connect(":memory:")
        original = sqlite3.connect(DB_FILE)
        original.backup(connection)
        original.close()
        for (table,) in connection.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'").fetchall():
            columns = connection.execute(f'PRAGMA table_info("{table}")').fetchall()
            for filler in (None, "???"):
                values = [None if (col[5] and "INT" in str(col[2]).upper()) else filler for col in columns]
                try:
                    connection.execute(f'INSERT INTO "{table}" VALUES ({",".join("?" * len(columns))})', values)
                except sqlite3.Error:
                    pass
        return connection, connection
    with open(LAB_FILE, newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.reader(handle))
    width = len(rows[0]) if rows else 1
    bad_path = f"{OUT}/test_bad_lab.csv"
    with open(bad_path, "w", newline="", encoding="utf-8") as handle:
        csv.writer(handle).writerows(rows + [[""] * width, ["???"] * width])
    return bad_path, None

ISO_DATE = re.compile(r"\d{4}-\d{2}-\d{2}(T\d{2}:\d{2}(:\d{2}(\.\d+)?)?([+-]\d{2}:\d{2}|Z)?)?")
PATIENT_REFERENCE = re.compile(r"Patient/[A-Za-z0-9\-]+")

def valid_iso(value):
    if not isinstance(value, str) or not ISO_DATE.fullmatch(value):
        return False
    try:
        dt.date.fromisoformat(value[:10])
        return True
    except ValueError:
        return False

# Spot checks against known correct answers (placeholder data only; Person 3 can add checks for team data)
def expect(records, resource, where, wanted):
    matches = [r for r in records if r.get("resourceType") == resource and all(get_path(r, p) == v for p, v in where.items())]
    if not matches:
        return False, f"no {resource} record found matching {where}"
    record = matches[0]
    wrong = []
    for path, value in wanted.items():
        actual = get_path(record, path)
        ok = (abs(float(actual) - value) < 1e-9 if isinstance(value, (int, float)) and isinstance(actual, (int, float)) and not isinstance(actual, bool)
              else isinstance(actual, str) and actual.startswith(str(value)))
        if not ok:
            wrong.append(f"{path} is {shape_mask(actual) if actual is not None else 'missing'}")
    return not wrong, "; ".join(wrong)

SPOT_CHECKS = {
    "patient_api": [("P001 date of birth, gender and surname converted correctly",
                     lambda recs: expect(recs, "Patient", {"id": "P001"}, {"birthDate": "1985-04-12", "gender": "female", "name[0].family": "Murphy"}))],
    "hospital_db": [("P001's cardiology appointment is booked for 2026-11-20 10:30",
                     lambda recs: expect(recs, "Appointment", {"participant[0].actor.reference": "Patient/P001", "serviceType[0].text": "Cardiology"},
                                         {"start": "2026-11-20T10:30", "status": "booked"})),
                    ("status code N (no-show) translated",
                     lambda recs: expect(recs, "Appointment", {"participant[0].actor.reference": "Patient/P004", "serviceType[0].text": "Cardiology"}, {"status": "noshow"}))],
    "lab_csv": [("P001 haemoglobin 13.5 g/dL on 2026-11-02",
                 lambda recs: expect(recs, "Observation", {"subject.reference": "Patient/P001", "code.coding[0].code": "718-7"},
                                     {"valueQuantity.value": 13.5, "valueQuantity.unit": "g/dL", "effectiveDateTime": "2026-11-02"}))],
} if DATA_SOURCE == "placeholder" else {}

def contract_tests(system_id, function, expected_resources):
    results = []
    def check(name, passed, detail=""):
        results.append({"test": name, "passed": bool(passed), "detail": "" if passed else redact(detail)})
    source, closer = open_source(system_id)
    records, logs, error = run_adapter(function, source)
    if closer:
        closer.close()
    check("runs without crashing", error is None, error or "")
    if error is not None:
        return results, [], logs
    check("returns a list of records", isinstance(records, list), f"returned {type(records).__name__}")
    if not isinstance(records, list):
        return results, [], logs
    check("returns at least one record", len(records) > 0, "returned an empty list")
    wrong_types = sorted({str(r.get("resourceType")) if isinstance(r, dict) else type(r).__name__ for r in records
                          if not isinstance(r, dict) or r.get("resourceType") not in expected_resources})
    check("only produces the approved resource types", not wrong_types, f"unexpected {wrong_types[:5]}; approved {sorted(expected_resources)}")
    records = [r for r in records if isinstance(r, dict)]
    missing, bad_dates, bad_refs, bad_codes, bad_numbers = [], [], [], [], []
    for r in records:
        resource, rid = r.get("resourceType"), r.get("id")
        for path in TARGET.get(resource, {}).get("required", []):
            if get_path(r, path) in (None, "", []):
                missing.append(f"{resource} {rid}: {path}")
        for path in DATE_FIELDS.get(resource, []):
            value = get_path(r, path)
            if value is not None and not valid_iso(value):
                bad_dates.append(f"{resource} {rid}: {path} looks like '{shape_mask(value)}'")
        for path in REFERENCE_FIELDS.get(resource, []):
            value = get_path(r, path)
            if value is not None and not (isinstance(value, str) and PATIENT_REFERENCE.fullmatch(value)):
                bad_refs.append(f"{resource} {rid}: {path} = '{shape_mask(value)}'")
        for (res, path), allowed in ALLOWED_VALUES.items():
            value = get_path(r, path) if res == resource else None
            if value is not None and value not in allowed:
                bad_codes.append(f"{resource} {rid}: {path} = '{value}'")
        for path in NUMBER_FIELDS.get(resource, []):
            value = get_path(r, path)
            if value is not None and (isinstance(value, bool) or not isinstance(value, (int, float))):
                bad_numbers.append(f"{resource} {rid}: {path} is {type(value).__name__}, not a number")
    ids = [(r.get("resourceType"), r.get("id")) for r in records]
    duplicates = sorted({f"{t} {i}" for t, i in ids if ids.count((t, i)) > 1})
    check("required fields are filled", not missing, "; ".join(missing[:6]))
    check("dates are ISO 8601", not bad_dates, "; ".join(bad_dates[:6]))
    check("patient links are valid references", not bad_refs, "; ".join(bad_refs[:6]))
    check("coded values use allowed words", not bad_codes, "; ".join(bad_codes[:6]))
    check("numeric values are numbers", not bad_numbers, "; ".join(bad_numbers[:6]))
    check("record IDs are unique", not duplicates, "duplicates: " + ", ".join(duplicates[:6]))
    bad_source, bad_closer = make_bad_source(system_id)
    bad_records, bad_logs, bad_error = run_adapter(function, bad_source)
    if bad_closer:
        bad_closer.close()
    check("handles broken records without crashing", bad_error is None, bad_error or "")
    if bad_error is None:
        count = len(bad_records) if isinstance(bad_records, list) else -1
        check("rejects broken records instead of passing them on", count == len(records),
              f"after adding 2 broken records the adapter returned {count} records (expected {len(records)})")
        check("reports each rejected record in the log", len(bad_logs) >= len(logs) + 2,
              f"only {len(bad_logs) - len(logs)} new log message(s) for 2 broken records")
    leaked = [m for m in logs + bad_logs if contains_phi(m)]
    check("no personal data in log messages", not leaked, f"{len(leaked)} log message(s) contained personal data")
    for name, spot_check in SPOT_CHECKS.get(system_id, []):
        try:
            ok, detail = spot_check(records)
        except Exception as error:
            ok, detail = False, f"{type(error).__name__}: {error}"
        check(f"spot check: {name}", ok, detail)
    return results, records, logs

generation = {}
for system_id in SYSTEM_IDS:
    heading(SYSTEM_LABELS[system_id])
    mappings = [m for m in design.get("mappings", []) if m.get("system_id") == system_id]
    expected_resources = {m.get("target", "").split(".")[0] for m in mappings} & set(TARGET)
    prompt = generation_prompt(system_id, mappings)
    history, agent_seconds, results = [], 0.0, []
    for attempt in range(1, MAX_FIX_ATTEMPTS + 1):
        reply, seconds = ask_agent(f"generate_{system_id}_attempt{attempt}", prompt)
        agent_seconds += seconds
        code = reply.get("code", "")
        code_path = f"{OUT}/generated/{system_id}_attempt{attempt}.py"
        with open(code_path, "w") as handle:
            handle.write(code)
        problems = safety_check(code, system_id)
        if problems:
            results = [{"test": "safety check before running", "passed": False, "detail": "; ".join(problems)}]
        else:
            namespace = {}
            try:
                exec(compile(code, code_path, "exec"), namespace)
                function = namespace[f"extract_{system_id}"]
                results, _, _ = contract_tests(system_id, function, expected_resources)
            except Exception as error:
                results = [{"test": "code loads", "passed": False, "detail": redact(f"{type(error).__name__}: {error}")}]
            results.insert(0, {"test": "safety check before running", "passed": True, "detail": ""})
        passed = sum(r["passed"] for r in results)
        history.append({"attempt": attempt, "checks_passed": passed, "checks_total": len(results),
                        "failed": [r["test"] for r in results if not r["passed"]]})
        audit("generation_attempt", system=system_id, attempt=attempt, checks_passed=passed, checks_total=len(results))
        print(f"   attempt {attempt}: {passed}/{len(results)} checks passed")
        if passed == len(results):
            break
        for failure in (r for r in results if not r["passed"]):
            print(f"     ✗ {failure['test']}: {failure['detail']}")
        if attempt < MAX_FIX_ATTEMPTS:
            print("   🔧 Self-healing: sending the failed checks back to the agent to fix its own code")
            prompt = heal_prompt(system_id, mappings, code, [r for r in results if not r["passed"]])
    shutil.copy(code_path, f"{OUT}/generated/{system_id}.py")
    all_passed = passed == len(results)
    generation[system_id] = {"attempts": attempt, "checks_passed": passed, "checks_total": len(results), "all_passed": all_passed,
                             "self_healed": all_passed and attempt > 1, "agent_seconds": round(agent_seconds, 1),
                             "code_file": f"{OUT}/generated/{system_id}.py", "explanation": reply.get("explanation", ""),
                             "suggested_tests": reply.get("suggested_tests", []), "history": history, "checks": results}
    display(pd.DataFrame(results)[["test", "passed", "detail"]])
    print("Agent's explanation:", reply.get("explanation", ""))

heading("Summary")
summary = pd.DataFrame([{"system": SYSTEM_LABELS[s], "attempts": g["attempts"], "checks passed": f"{g['checks_passed']}/{g['checks_total']}",
                         "self-healed": "yes" if g["self_healed"] else "no", "agent time (s)": g["agent_seconds"], "code file": g["code_file"]}
                        for s, g in generation.items()])
display(summary)
with open(f"{OUT}/3_generation_summary.json", "w") as handle:
    json.dump(generation, handle, indent=2, ensure_ascii=False)
with open(f"{OUT}/3_suggested_tests.md", "w") as handle:
    handle.write("# Tests suggested by the agent (for Person 3)\n")
    for s, g in generation.items():
        handle.write(f"\n## {SYSTEM_LABELS[s]}\n" + "".join(f"- {t}\n" for t in g["suggested_tests"]))
for s in SYSTEM_IDS:
    heading(f"Generated code · {SYSTEM_LABELS[s]}")
    print(open(f"{OUT}/generated/{s}.py").read())
stage_finished("generate", agent_seconds=round(sum(g["agent_seconds"] for g in generation.values()), 1),
               all_passed=all(g["all_passed"] for g in generation.values()))

In [ ]:
#@title ✅ Human review · Generated code { display-mode: "form" }
#@markdown Approve only if every check passed and the code reads sensibly. Nothing runs against the systems for real until this is approved.
REVIEWER = "Jewel"  #@param {type:"string"}
APPROVED = True  #@param {type:"boolean"}
NOTES = ""  #@param {type:"string"}
if APPROVED and not STAGES.get("generate", {}).get("all_passed"):
    raise RuntimeError("Some checks are still failing. Re-run stage 3, or fix the design and regenerate, before approving.")
approve("generate", REVIEWER, APPROVED, NOTES)

---
## Stage 4 · Operate: the integration running
**What happens:** the approved adapters run against all three systems and load everything into one store. The **common API** then answers `GET /Patient/{id}/$everything` with one combined FHIR record. Problems such as rejected rows or results for unknown patients appear as **alerts**, and a privacy scan checks that no personal data reached the logs.

In [ ]:
#@title 9 · STAGE 4: Operate · run the integration and call the common API { display-mode: "form" }
#@markdown Which patient to show in the demo:
DEMO_PATIENT_ID = "P001"  #@param {type:"string"}
#@markdown (If this ID isn't in the data, the notebook picks a patient with records in all three systems.)
require_approved("generate")

ALERTS = []
def alert(level, message):
    entry = {"time": dt.datetime.now().isoformat(timespec="seconds"), "level": level, "message": redact(message)}
    ALERTS.append(entry)
    audit("alert", level=level, message=entry["message"])

ADAPTERS = {}
for system_id in SYSTEM_IDS:
    namespace = {}
    exec(compile(open(f"{OUT}/generated/{system_id}.py").read(), f"{system_id}.py", "exec"), namespace)
    ADAPTERS[system_id] = namespace[f"extract_{system_id}"]

STORE, run_rows = {}, []
for system_id, adapter in ADAPTERS.items():
    source, closer = open_source(system_id)
    started = time.time()
    records, logs, error = run_adapter(adapter, source)
    if closer:
        closer.close()
    seconds = round(time.time() - started, 3)
    if error:
        alert("ERROR", f"{system_id}: integration failed: {error}")
        records = []
    for message in logs:
        alert("WARNING", f"{system_id}: {message}")
    for record in records:
        STORE.setdefault(record["resourceType"], []).append(record)
    run_rows.append({"system": SYSTEM_LABELS[system_id], "records loaded": len(records), "records rejected": len(logs), "status": "ERROR" if error else "OK", "seconds": seconds})
    audit("integration_run", system=system_id, records=len(records), rejected=len(logs), status="ERROR" if error else "OK", seconds=seconds)

known_patients = {p["id"] for p in STORE.get("Patient", [])}
for resource, records in STORE.items():
    for record in records:
        for path in REFERENCE_FIELDS.get(resource, []):
            reference = get_path(record, path) or ""
            if reference and reference.split("/")[-1] not in known_patients:
                alert("WARNING", f"{resource} {record.get('id')} refers to {reference}, which is not in the registration system. Held for review.")

def get_patient_everything(patient_id):
    """The common API: GET /Patient/{id}/$everything (FHIR searchset Bundle)."""
    audit("common_api_call", path=f"/Patient/{patient_id}/$everything")
    entries = [r for r in STORE.get("Patient", []) if r.get("id") == patient_id]
    if not entries:
        raise LookupError(f"404: Patient {patient_id} not found")
    reference = f"Patient/{patient_id}"
    for resource, records in STORE.items():
        entries += [r for r in records if any(get_path(r, p) == reference for p in REFERENCE_FIELDS.get(resource, []))]
    return {"resourceType": "Bundle", "type": "searchset", "total": len(entries), "entry": [{"resource": r} for r in entries]}

heading("Integration run")
display(pd.DataFrame(run_rows))
if DEMO_PATIENT_ID not in known_patients:
    linked = {}
    for resource, records in STORE.items():
        for record in records:
            for path in REFERENCE_FIELDS.get(resource, []):
                pid = (get_path(record, path) or "").split("/")[-1]
                if pid in known_patients:
                    linked[pid] = linked.get(pid, 0) + 1
    if linked:
        DEMO_PATIENT_ID = max(sorted(linked), key=lambda pid: linked[pid] if linked[pid] <= 40 else 40 - linked[pid] / 1000)
        print(f"(Patient in the settings isn't in this data, so showing {DEMO_PATIENT_ID}, a patient with records in every system.)")
heading(f"Common API · GET /Patient/{DEMO_PATIENT_ID}/$everything")
bundle = get_patient_everything(DEMO_PATIENT_ID)
with open(f"{OUT}/4_patient_{DEMO_PATIENT_ID}_bundle.json", "w") as handle:
    json.dump(bundle, handle, indent=2, ensure_ascii=False)
resources = [e["resource"] for e in bundle["entry"]]
patient = resources[0]
print(f"👤 {' '.join(get_path(patient, 'name[0].given') or [])} {get_path(patient, 'name[0].family')} · born {patient.get('birthDate')} · {patient.get('gender')} · ID {patient.get('id')}   (from the patient API)")
appointments = [{"when": r.get("start"), "department": get_path(r, "serviceType[0].text"), "status": r.get("status"), "id": r.get("id"), "came from": "hospital database"}
                for r in resources if r.get("resourceType") == "Appointment"]
encounters = [{"start": get_path(r, "period.start"), "end": get_path(r, "period.end"), "status": r.get("status"), "id": r.get("id"), "came from": "hospital database"}
              for r in resources if r.get("resourceType") == "Encounter"]
labs = [{"date": r.get("effectiveDateTime"), "test": get_path(r, "code.text") or get_path(r, "code.coding[0].display"),
         "result": f"{get_path(r, 'valueQuantity.value')} {get_path(r, 'valueQuantity.unit')}", "normal range": get_path(r, "referenceRange[0].text"), "came from": "lab CSV"}
        for r in resources if r.get("resourceType") == "Observation"]
if appointments:
    print("\n📅 Appointments"); display(pd.DataFrame(appointments))
if encounters:
    print("\n🏥 Visits and admissions"); display(pd.DataFrame(encounters))
if labs:
    print("\n🧪 Lab results"); display(pd.DataFrame(labs))
print("\nRaw FHIR Bundle (first 1,500 characters):")
print(json.dumps(bundle, indent=2, ensure_ascii=False)[:1500])

heading("Alerts for the operations team")
display(pd.DataFrame(ALERTS) if ALERTS else "No alerts.")
with open(f"{OUT}/4_alerts.json", "w") as handle:
    json.dump(ALERTS, handle, indent=2, ensure_ascii=False)

heading("🛡️ Privacy scan of all logs")
log_lines = open(AUDIT_FILE, encoding="utf-8").read().splitlines()
leaks = [line for line in log_lines if contains_phi(line)]
print(f"Scanned {len(log_lines)} log lines for {len(PHI_VALUES)} known personal values (names, dates of birth, phone numbers): {len(leaks)} found.")
print("✅ No personal data in the logs." if not leaks else "❌ Personal data found in the logs. Investigate before going any further.")
stage_finished("operate", records=sum(len(v) for v in STORE.values()), alerts=len(ALERTS), log_leaks=len(leaks))

---
## Results · AI-assisted vs manual
Person 3 fills in the manual numbers below. If Person 3 saves their hand-written adapters as `manual/patient_api.py`, `manual/hospital_db.py` and `manual/lab_csv.py` (same function names: `extract_patient_api(source, log)` and so on), **the exact same automated checks are run on them**, which makes it a fair comparison.

In [ ]:
#@title 10 · Results: time, correctness, consistency and reliability { display-mode: "form" }
#@markdown **Person 3's manual numbers** (time includes reading the systems, writing, testing and fixing):
MANUAL_MINUTES = 53  #@param {type:"number"}
MANUAL_FIELDS_CORRECT = 16  #@param {type:"integer"}
MANUAL_FIELDS_TOTAL = 16  #@param {type:"integer"}
MANUAL_NOTES = "Code written by Claude, reviewed and run by Person 3"  #@param {type:"string"}

# ---- mapping correctness against an answer key (placeholder data; Person 3 writes one for team data) ----
EXPECTED_MAPPINGS = [
    ("patient_api", "Patient.id", "patient_id"), ("patient_api", "Patient.name[0].family", "full_name"),
    ("patient_api", "Patient.name[0].given", "full_name"), ("patient_api", "Patient.birthDate", "dob"),
    ("patient_api", "Patient.gender", "sex"),
    ("hospital_db", "Appointment.id", "APPT_NO"), ("hospital_db", "Appointment.status", "STAT"),
    ("hospital_db", "Appointment.start", "APPT_DT"), ("hospital_db", "Appointment.serviceType[0].text", "DEPT"),
    ("hospital_db", "Appointment.participant[0].actor.reference", "PT_ID"),
    ("lab_csv", "Observation.subject.reference", "patient_ref"), ("lab_csv", "Observation.effectiveDateTime", "sample_date"),
    ("lab_csv", "Observation.code.coding[0].code", "test_code"), ("lab_csv", "Observation.code.text", "test_name"),
    ("lab_csv", "Observation.valueQuantity.value", "result"), ("lab_csv", "Observation.valueQuantity.unit", "units"),
    ("lab_csv", "Observation.referenceRange[0].text", "ref_range"),
] if DATA_SOURCE == "placeholder" else []
if DATA_SOURCE == "team" and os.path.exists("team_data/answer_key.json"):
    EXPECTED_MAPPINGS = [tuple(row) for row in json.load(open("team_data/answer_key.json"))]

def path_tokens(path):
    return [t.lower() for t in re.findall(r"[A-Za-z]+", path)]

def mapping_found(system_id, target, source_hint):
    resource, path = target.split(".", 1)
    for m in design.get("mappings", []):
        t = str(m.get("target", ""))
        if m.get("system_id") == system_id and t.lower().startswith(resource.lower() + ".") \
                and path_tokens(t.split(".", 1)[1]) == path_tokens(path) and source_hint.lower() in str(m.get("source", "")).lower():
            return True
    return False

mapping_rows = [{"system": s, "target": t, "expected source": h, "agent got it right": mapping_found(s, t, h)} for s, t, h in EXPECTED_MAPPINGS]
mapping_score = f"{sum(r['agent got it right'] for r in mapping_rows)}/{len(mapping_rows)}" if mapping_rows else "add an answer key"

# ---- consistency: does every adapter follow the same template and rules? ----
def consistency_checks(code, system_id):
    tree = ast.parse(code)
    functions = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
    main = next((f for f in functions if f.name == f"extract_{system_id}"), None)
    calls = {n.func.id for n in ast.walk(tree) if isinstance(n, ast.Call) and isinstance(n.func, ast.Name)}
    return {
        "standard name and signature": bool(main) and [a.arg for a in main.args.args] == ["source", "log"],
        "has a docstring": bool(main) and ast.get_docstring(main) is not None,
        "reports problems through log()": "log" in calls,
        "no print statements": "print" not in calls,
        "passes the safety check": not safety_check(code, system_id),
    }

def score_adapters(folder):
    rows, totals = [], [0, 0, 0, 0]
    for system_id in SYSTEM_IDS:
        path = f"{folder}/{system_id}.py"
        if not os.path.exists(path):
            return None, None
        code = open(path).read()
        consistency = consistency_checks(code, system_id)
        namespace = {}
        exec(compile(code, path, "exec"), namespace)
        mappings = [m for m in design.get("mappings", []) if m.get("system_id") == system_id]
        expected = {m.get("target", "").split(".")[0] for m in mappings} & set(TARGET)
        results, _, _ = contract_tests(system_id, namespace[f"extract_{system_id}"], expected)
        totals[0] += sum(consistency.values()); totals[1] += len(consistency)
        totals[2] += sum(r["passed"] for r in results); totals[3] += len(results)
        rows.append({"system": SYSTEM_LABELS[system_id], **consistency, "checks passed": f"{sum(r['passed'] for r in results)}/{len(results)}"})
    return pd.DataFrame(rows), totals

ai_table, ai_totals = score_adapters(f"{OUT}/generated")
manual_table, manual_totals = score_adapters("manual")

agent_seconds = sum(STAGES.get(s, {}).get("agent_seconds", 0) for s in ("discover", "design", "generate"))
review_seconds = sum(STAGES.get(s, {}).get("review_seconds", 0) for s in ("discover", "design", "generate"))
ai_minutes = (agent_seconds + review_seconds) / 60
attempts = sum(g["attempts"] for g in generation.values())

comparison = pd.DataFrame([
    {"measure": "Time to build all 3 integrations (minutes)",
     "manual": MANUAL_MINUTES or "Person 3 to fill in",
     "AI-assisted": f"{ai_minutes:.1f} (agent {agent_seconds / 60:.1f} + human review {review_seconds / 60:.1f})"},
    {"measure": "Correctness: key fields mapped correctly",
     "manual": f"{MANUAL_FIELDS_CORRECT}/{MANUAL_FIELDS_TOTAL}" if MANUAL_FIELDS_TOTAL else "Person 3 to fill in",
     "AI-assisted": mapping_score},
    {"measure": "Consistency: template and rule checks passed (all adapters)",
     "manual": f"{manual_totals[0]}/{manual_totals[1]}" if manual_totals else "save manual adapters in /manual",
     "AI-assisted": f"{ai_totals[0]}/{ai_totals[1]}"},
    {"measure": "Reliability: automated checks passed, incl. broken-data handling",
     "manual": f"{manual_totals[2]}/{manual_totals[3]}" if manual_totals else "save manual adapters in /manual",
     "AI-assisted": f"{ai_totals[2]}/{ai_totals[3]}"},
    {"measure": "Agent attempts / self-healing fixes", "manual": "n/a",
     "AI-assisted": f"{attempts} attempts across 3 adapters · {sum(g['self_healed'] for g in generation.values())} fixed itself"},
    {"measure": "Personal data found in logs", "manual": "n/a", "AI-assisted": STAGES.get("operate", {}).get("log_leaks", "run stage 4")},
])
heading("AI-assisted vs manual")
display(comparison)
if mapping_rows:
    heading("Mapping answer key")
    display(pd.DataFrame(mapping_rows))
heading("Consistency of the generated adapters")
display(ai_table)
if manual_table is not None:
    heading("Consistency of the manual adapters")
    display(manual_table)
heading("Time per stage")
display(pd.DataFrame([{"stage": s, "agent seconds": STAGES.get(s, {}).get("agent_seconds"), "review seconds": STAGES.get(s, {}).get("review_seconds"),
                       "approved by": STAGES.get(s, {}).get("reviewer")} for s in ("discover", "design", "generate")]))

comparison.to_csv(f"{OUT}/5_results_comparison.csv", index=False)
with open(f"{OUT}/5_results.json", "w") as handle:
    json.dump({"run": RUN_ID, "mode": MODE, "model": MODEL_NAME, "data": DATA_SOURCE, "stages": STAGES,
               "mapping_score": mapping_score, "manual_notes": MANUAL_NOTES}, handle, indent=2, default=str)
shutil.make_archive("exl_agent_outputs", "zip", OUT)
print("📦 All outputs zipped: exl_agent_outputs.zip")
try:
    from google.colab import files
    files.download("exl_agent_outputs.zip")
except Exception:
    pass

---
### For the deck and the submission
- **AI tools declaration:** the agent runs on Google Gemini (model shown in cell 5). This notebook's code and the built-in example outputs were written with the help of Claude (Anthropic). Declare both.
- **Responsible AI evidence:** synthetic data only · personal fields masked before anything reaches the AI model · generated code safety-checked before it runs · human sign-off required at every stage (`audit_log.jsonl` shows who approved what, and when) · broken records rejected and alerted, never guessed · privacy scan of the logs.
- **Production path:** the same pattern on Azure, with Azure OpenAI as the model, .NET adapters generated from the same template, Azure API Management as the gateway and Azure Monitor for alerts.